# SmartEats — Multi-Model AI System

**Assignment 7: Multi-Model AI Systems**

This notebook contains the experiments, benchmarking results, and analysis for all parts of the assignment.


In [1]:
import pandas as pd
from collections import defaultdict
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 400)


## Part 1: Model Size Selection and Benchmarking

### Step 1.1: Model Size Categories

We selected 15 models from the Hugging Face leaderboard and organized them into five size categories based on parameter counts. Each category contains exactly 3 models.

| Model Category | Parameter Range | Models |
|---|---|---|
| Ultra-Light | < 1B | Qwen/Qwen2.5-0.5B-Instruct, HuggingFaceTB/SmolLM2-360M-Instruct, EleutherAI/pythia-410m |
| Small | 1B – 3B | stabilityai/stablelm-2-zephyr-1_6b, microsoft/phi-2, TinyLlama/TinyLlama-1.1B-Chat-v1.0 |
| Medium | 3B – 7B | stabilityai/stablelm-zephyr-3b, microsoft/Phi-3-mini-4k-instruct, Qwen/Qwen2.5-3B-Instruct |
| Large | 7B | HuggingFaceH4/zephyr-7b-beta, NousResearch/Nous-Hermes-2-Mistral-7B-DPO, tiiuae/falcon-7b-instruct |
| Extra-Large | 7B – 10.7B+ | 01-ai/Yi-1.5-9B-Chat, lmsys/vicuna-7b-v1.5, upstage/SOLAR-10.7B-Instruct-v1.0 |


In [2]:
models_by_category = {
    "Ultra-Light (< 1B)": [
        ("Qwen/Qwen2.5-0.5B-Instruct", "0.5B"),
        ("HuggingFaceTB/SmolLM2-360M-Instruct", "360M"),
        ("EleutherAI/pythia-410m", "410M"),
    ],
    "Small (1B – 3B)": [
        ("stabilityai/stablelm-2-zephyr-1_6b", "1.6B"),
        ("microsoft/phi-2", "2.7B"),
        ("TinyLlama/TinyLlama-1.1B-Chat-v1.0", "1.1B"),
    ],
    "Medium (3B – 7B)": [
        ("stabilityai/stablelm-zephyr-3b", "3B"),
        ("microsoft/Phi-3-mini-4k-instruct", "3.8B"),
        ("Qwen/Qwen2.5-3B-Instruct", "3B"),
    ],
    "Large (7B)": [
        ("HuggingFaceH4/zephyr-7b-beta", "7B"),
        ("NousResearch/Nous-Hermes-2-Mistral-7B-DPO", "7B"),
        ("tiiuae/falcon-7b-instruct", "7B"),
    ],
    "Extra-Large (7B – 10.7B+)": [
        ("01-ai/Yi-1.5-9B-Chat", "9B"),
        ("lmsys/vicuna-7b-v1.5", "7B"),
        ("upstage/SOLAR-10.7B-Instruct-v1.0", "10.7B"),
    ],
}

rows = []
for cat, models in models_by_category.items():
    for name, params in models:
        rows.append({"Category": cat, "Model": name, "Parameters": params})

df_models = pd.DataFrame(rows)
print(f"Total models: {len(df_models)}")
print(f"Categories: {df_models['Category'].nunique()}")
print()
df_models


Total models: 15
Categories: 5



,Category,Model,Parameters
0,Ultra-Light (< 1B),Qwen/Qwen2.5-0.5B-Instruct,0.5B
1,Ultra-Light (< 1B),HuggingFaceTB/SmolLM2-360M-Instruct,360M
2,Ultra-Light (< 1B),EleutherAI/pythia-410m,410M
3,Small (1B – 3B),stabilityai/stablelm-2-zephyr-1_6b,1.6B
4,Small (1B – 3B),microsoft/phi-2,2.7B
5,Small (1B – 3B),TinyLlama/TinyLlama-1.1B-Chat-v1.0,1.1B
6,Medium (3B – 7B),stabilityai/stablelm-zephyr-3b,3B
7,Medium (3B – 7B),microsoft/Phi-3-mini-4k-instruct,3.8B
8,Medium (3B – 7B),Qwen/Qwen2.5-3B-Instruct,3B
9,Large (7B),HuggingFaceH4/zephyr-7b-beta,7B


### Step 1.2: Prompt Types

We defined 5 prompt types that match SmartEats' AI tasks — a campus dining and meal-tracking app that uses LLMs for nutrition analysis, user profiling, and personalized recommendations.

| # | Prompt Type | Description |
|---|---|---|
| 1 | Classification | Categorize users by fitness goal, eating habits, food preferences, or engagement level |
| 2 | Prediction | Forecast weight trends, calorie balance, meal-logging retention, and goal attainment risk |
| 3 | Structured Extraction | Parse free-text user notes into structured key-value pairs for profiles |
| 4 | Summarization | Condense meal logs, progress data, and profiles into one-sentence summaries |
| 5 | Text Generation | Generate personalized greetings, motivational messages, and health tips |


### Step 1.3: Evaluation Prompts

5 prompts per prompt type × 5 types = **25 total prompts**. Each prompt simulates a real input that SmartEats would receive.


In [3]:
prompts_by_type = {
    "classification": [
        "Based on the selected user's UserProfile (height_cm, weight_kg, body_fat_pct, age, activity_level), classify their most suitable fitness goal. Possible labels: fat_loss, muscle_gain, maintain. Return only the label.",
        "Based on the selected user's UserProfile (weight_kg, height_cm, body_fat_pct, activity_level, goal), determine whether their stated goal is appropriate for their current physical condition. Possible labels: match, mismatch. Return only the label.",
        "Based on the selected user's MealLog records, classify their eating habit by examining which meal_type entries (breakfast, lunch, dinner) appear or are missing across dates. Possible labels: regular_three_meals, skip_breakfast, skip_lunch, skip_dinner, irregular_meals. Return only the label.",
        "Based on the selected user's MealLog records, classify their food preference by examining the meal_name entries. Possible labels: asian_only, western_only, salad_heavy, mixed. Return only the label.",
        "Based on the selected user's MealLog records, classify their app engagement level by counting meals logged per day and the number of distinct dates. Possible labels: power_user, regular_user, casual_user, inactive. Return only the label.",
    ],
    "prediction": [
        "Based on the selected user's WeightLog history, BodyFatLog history, and UserProfile (goal, activity_level), predict their weight trend over the next 30 days. Possible labels: weight_loss, weight_gain, stable. Return only the label.",
        "Based on the selected user's MealLog (calories, protein_g, carbs_g, fat_g per meal) and UserProfile (goal, activity_level), predict whether their current diet will lead to a calorie surplus or deficit over the next 7 days. Possible labels: surplus, deficit, balanced. Return only the label.",
        "Based on the selected user's MealLog records, predict whether this user is likely to continue logging meals next week, based on the frequency and recency of past logs. Possible labels: will_continue, might_stop, already_stopped. Return only the label.",
        "Based on the selected user's WeightLog trend, BodyFatLog trend, and UserProfile (goal, activity_level), predict whether they are at risk of not achieving their goal within 3 months. Possible labels: on_track, at_risk, off_track. Return only the label.",
        "Based on the selected user's MealLog records, predict whether their eating habit is likely to change in the next month, based on patterns in meal_type, meal_name, and timestamps. Possible labels: stable_habit, likely_change, unclear. Return only the label.",
    ],
    "structured_extraction": [
        "From the selected user's UserNote, extract structured information about their eating preferences. Return as key-value pairs.",
        "From the selected user's UserNote, extract structured information about their activity level and exercise habits. Return as key-value pairs.",
        "From the selected user's UserNote, extract structured information about their fitness goal, including target amount and timeline if mentioned. Return as key-value pairs.",
        "From the selected user's UserNote, extract any dietary restrictions mentioned. Return as key-value pairs.",
        "From the selected user's UserNote, extract body condition information that may affect their exercise or diet plan. Return as key-value pairs.",
    ],
    "summarization": [
        "Based on the selected user's MealLog records, summarize their overall eating behavior in one sentence, covering meal timing, calorie intake, and nutritional balance.",
        "Based on the selected user's UserProfile (weight_kg, height_cm, body_fat_pct, activity_level, goal), provide a one-line summary of whether their stated goal is realistic and what adjustments may help.",
        "Based on the selected user's MealLog records, summarize their diet pattern in one sentence, focusing on meal frequency, meal types, and any noticeable trends.",
        "Based on the selected user's MealLog records and BodyFatLog history, summarize their level of progress and consistency in one sentence.",
        "Based on the selected user's WeightLog history, BodyFatLog history, and UserProfile (goal, activity_level), summarize their overall progress toward their fitness goal in one sentence.",
    ],
    "text_generation": [
        "Generate a personalised good morning greeting for a user who is on a fat loss goal.",
        "Generate a short motivational message for a user who has been consistently logging meals but has not seen weight change in 2 weeks.",
        "Generate one practical healthy eating tip suitable for a university student with a busy schedule.",
        "Generate a short reminder message for a user who has not logged any meals in the past 3 days.",
        "Generate one follow-up question to better understand why a user keeps skipping breakfast.",
    ],
}

for pt, prompts in prompts_by_type.items():
    print(f"\n{'='*70}")
    print(f"  {pt.upper()} ({len(prompts)} prompts)")
    print(f"{'='*70}")
    for i, p in enumerate(prompts, 1):
        print(f"  {i}. {p}")

total = sum(len(v) for v in prompts_by_type.values())
print(f"\nTotal prompts: {total}")



  CLASSIFICATION (5 prompts)
  1. Based on the selected user's UserProfile (height_cm, weight_kg, body_fat_pct, age, activity_level), classify their most suitable fitness goal. Possible labels: fat_loss, muscle_gain, maintain. Return only the label.
  2. Based on the selected user's UserProfile (weight_kg, height_cm, body_fat_pct, activity_level, goal), determine whether their stated goal is appropriate for their current physical condition. Possible labels: match, mismatch. Return only the label.
  3. Based on the selected user's MealLog records, classify their eating habit by examining which meal_type entries (breakfast, lunch, dinner) appear or are missing across dates. Possible labels: regular_three_meals, skip_breakfast, skip_lunch, skip_dinner, irregular_meals. Return only the label.
  4. Based on the selected user's MealLog records, classify their food preference by examining the meal_name entries. Possible labels: asian_only, western_only, salad_heavy, mixed. Return only the la

### Step 1.4: Evaluation Metrics

We defined the following metrics for self-evaluation. Each metric is scored 1–10. The final Self-Evaluation Score is the simple average across all applicable metrics (some metrics are skipped for certain prompt types — e.g., "Tone" is excluded for classification tasks).

| Metric | Description |
|---|---|
| Instruction Adherence | Did the model follow instructions? |
| Accuracy | Was the answer correct? |
| Clarity | Was the output understandable? |
| Structure | Did the output follow the expected format? |
| Tone | How was the tone of the response? (excluded for classification) |
| Match | Does the model's reasoning align with its final response? |
| Reasonable | Is the model's reasoning logical and sensible? |


### Step 1.5: Benchmark Experiments

Each of the 15 models was evaluated on all 25 prompts, producing **375 benchmark rows**. Each row includes the model, its category, tokens/sec, latency, prompt type, prompt text, and a self-evaluation score (1–10).


In [4]:
BENCHMARK_DATA = [
    ("Qwen/Qwen2.5-0.5B-Instruct", "Ultra-Light Models", 133, 2.07, "classification", "Based on the selected user's UserProfile (height_cm, weight_kg, body_fat_pct, age, activity_level), classify their most suitable fitness goal. Possible labels: fat_loss, muscle_gain, maintain. Return only the label.", 3.1667, {"instr_adhere": 3, "accuracy": 1, "clarity": 4, "structure": 5, "match": 3, "reasonable": 3}),
    ("Qwen/Qwen2.5-0.5B-Instruct", "Ultra-Light Models", 149, 1.88, "classification", "Based on the selected user's UserProfile (weight_kg, height_cm, body_fat_pct, activity_level, goal), determine whether their stated goal is appropriate for their current physical condition. Possible labels: match, mismatch. Return only the label.", 3.6667, {"instr_adhere": 4, "accuracy": 3, "clarity": 4, "structure": 4, "match": 4, "reasonable": 3}),
    ("Qwen/Qwen2.5-0.5B-Instruct", "Ultra-Light Models", 164, 1.88, "classification", "Based on the selected user's MealLog records, classify their eating habit by examining which meal_type entries (breakfast, lunch, dinner) appear or are missing across dates. Possible labels: regular_three_meals, skip_breakfast, skip_lunch, skip_dinner, irregular_meals. Return only the label.", 3.5, {"instr_adhere": 4, "accuracy": 1, "clarity": 4, "structure": 4, "match": 4, "reasonable": 4}),
    ("Qwen/Qwen2.5-0.5B-Instruct", "Ultra-Light Models", 93, 1, "classification", "Based on the selected user's MealLog records, classify their food preference by examining the meal_name entries. Possible labels: asian_only, western_only, salad_heavy, mixed. Return only the label.", 3, {"instr_adhere": 4, "accuracy": 2, "clarity": 4, "structure": 4, "match": 2, "reasonable": 2}),
    ("Qwen/Qwen2.5-0.5B-Instruct", "Ultra-Light Models", 147, 1.88, "classification", "Based on the selected user's MealLog records, classify their app engagement level by counting meals logged per day and the number of distinct dates. Possible labels: low_engagement, moderate_engagement, high_engagement. Return only the label.", 3.1667, {"instr_adhere": 4, "accuracy": 3, "clarity": 4, "structure": 4, "match": 3, "reasonable": 1}),
    ("Qwen/Qwen2.5-0.5B-Instruct", "Ultra-Light Models", 145, 1.88, "prediction", "Based on the selected user's WeightLog history, BodyFatLog history, and UserProfile (goal, activity_level), predict their weight trend over the next 30 days. Return a short prediction with expected direction and estimated change.", 3.6667, {"instr_adhere": 4, "accuracy": 4, "clarity": 4, "structure": 4, "match": 3, "reasonable": 3}),
    ("Qwen/Qwen2.5-0.5B-Instruct", "Ultra-Light Models", 154, 1.88, "prediction", "Based on the selected user's MealLog (calories, protein_g, carbs_g, fat_g per meal) and UserProfile (goal, activity_level), predict whether their current intake pattern will support their goal over the next 30 days. Return a short prediction.", 2.6667, {"instr_adhere": 3, "accuracy": 3, "clarity": 3, "structure": 3, "match": 2, "reasonable": 2}),
    ("Qwen/Qwen2.5-0.5B-Instruct", "Ultra-Light Models", 147, 1.87, "prediction", "Based on the selected user's MealLog records, predict whether this user is likely to continue logging meals next week, based on the frequency and recency of their entries. Return: likely or unlikely, with a one-line reason.", 3.3333, {"instr_adhere": 4, "accuracy": 3, "clarity": 4, "structure": 4, "match": 2, "reasonable": 3}),
    ("Qwen/Qwen2.5-0.5B-Instruct", "Ultra-Light Models", 150, 1.88, "prediction", "Based on the selected user's WeightLog trend, BodyFatLog trend, and UserProfile (goal, activity_level), predict whether they are at risk of not achieving their fitness goal. Return: at_risk or on_track, with a one-line reason.", 3.6667, {"instr_adhere": 4, "accuracy": 4, "clarity": 4, "structure": 4, "match": 3, "reasonable": 3}),
    ("Qwen/Qwen2.5-0.5B-Instruct", "Ultra-Light Models", 139, 1.88, "prediction", "Based on the selected user's MealLog records, predict whether their eating habit is likely to change in the next month, based on patterns in meal_type and meal_name. Return a short prediction.", 3.6667, {"instr_adhere": 4, "accuracy": 4, "clarity": 4, "structure": 4, "match": 3, "reasonable": 3}),
    ("Qwen/Qwen2.5-0.5B-Instruct", "Ultra-Light Models", 120, 1.87, "structured_extraction", "From the selected user's UserNote, extract structured information about their eating preferences. Return as key-value pairs.", 2.6667, {"instr_adhere": 2, "clarity": 3, "structure": 3}),
    ("Qwen/Qwen2.5-0.5B-Instruct", "Ultra-Light Models", 124, 1.86, "structured_extraction", "From the selected user's UserNote, extract structured information about their activity level and exercise habits. Return as key-value pairs.", 4, {"instr_adhere": 4, "clarity": 4, "structure": 4}),
    ("Qwen/Qwen2.5-0.5B-Instruct", "Ultra-Light Models", 129, 1.87, "structured_extraction", "From the selected user's UserNote, extract structured information about their fitness goal, including target amount and timeline if mentioned. Return as key-value pairs.", 4, {"instr_adhere": 4, "clarity": 4, "structure": 4}),
    ("Qwen/Qwen2.5-0.5B-Instruct", "Ultra-Light Models", 95, 1.43, "structured_extraction", "From the selected user's UserNote, extract any dietary restrictions mentioned. Return as key-value pairs.", 1, {"instr_adhere": 0, "clarity": 2, "structure": 1}),
    ("Qwen/Qwen2.5-0.5B-Instruct", "Ultra-Light Models", 126, 1.86, "structured_extraction", "From the selected user's UserNote, extract body condition information that may affect their exercise or diet plan. Return as key-value pairs.", 2, {"instr_adhere": 1, "clarity": 3, "structure": 2}),
    ("Qwen/Qwen2.5-0.5B-Instruct", "Ultra-Light Models", 130, 1.87, "summarization", "Based on the selected user's MealLog records, summarize their overall eating behavior in one sentence, covering meal timing, calorie intake, and nutritional balance.", 4, {"instr_adhere": 4, "clarity": 4, "structure": 4, "tone": 4}),
    ("Qwen/Qwen2.5-0.5B-Instruct", "Ultra-Light Models", 143, 1.88, "summarization", "Based on the selected user's UserProfile (weight_kg, height_cm, body_fat_pct, activity_level, goal), provide a one-line summary of whether their stated goal is realistic given their current body metrics.", 4, {"instr_adhere": 4, "clarity": 4, "structure": 4, "tone": 4}),
    ("Qwen/Qwen2.5-0.5B-Instruct", "Ultra-Light Models", 132, 1.88, "summarization", "Based on the selected user's MealLog records, summarize their diet pattern in one sentence, focusing on meal frequency, meal types, and any noticeable nutritional trends.", 3.25, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 4}),
    ("Qwen/Qwen2.5-0.5B-Instruct", "Ultra-Light Models", 126, 1.88, "summarization", "Based on the selected user's MealLog records and BodyFatLog history, summarize their level of progress and consistency in one sentence.", 3.75, {"instr_adhere": 4, "clarity": 4, "structure": 4, "tone": 3}),
    ("Qwen/Qwen2.5-0.5B-Instruct", "Ultra-Light Models", 135, 1.88, "summarization", "Based on the selected user's WeightLog history, BodyFatLog history, and UserProfile (goal, activity_level), summarize their overall progress toward their fitness goal in one sentence.", 3, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 3}),
    ("Qwen/Qwen2.5-0.5B-Instruct", "Ultra-Light Models", 116, 1.87, "text_generation", "Generate a personalised good morning greeting for a user who is on a fat loss goal.", 2, {"instr_adhere": 2, "clarity": 2, "structure": 2, "tone": 2}),
    ("Qwen/Qwen2.5-0.5B-Instruct", "Ultra-Light Models", 124, 1.86, "text_generation", "Generate a short motivational message for a user who has been consistently logging meals but has not seen weight change in 2 weeks.", 3, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 3}),
    ("Qwen/Qwen2.5-0.5B-Instruct", "Ultra-Light Models", 115, 1.86, "text_generation", "Generate one practical healthy eating tip suitable for a university student with a busy schedule.", 3, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 3}),
    ("Qwen/Qwen2.5-0.5B-Instruct", "Ultra-Light Models", 120, 1.86, "text_generation", "Generate a short reminder message for a user who has not logged any meals in the past 3 days.", 2.75, {"instr_adhere": 3, "clarity": 2, "structure": 3, "tone": 3}),
    ("Qwen/Qwen2.5-0.5B-Instruct", "Ultra-Light Models", 114, 1.85, "text_generation", "Generate one follow-up question to better understand why a user keeps skipping breakfast.", 2, {"instr_adhere": 2, "clarity": 2, "structure": 2, "tone": 2}),
    ("HuggingFaceTB/SmolLM2-360M-Instruct", "Ultra-Light Models", 154, 2.33, "classification", "Based on the selected user's UserProfile (height_cm, weight_kg, body_fat_pct, age, activity_level), classify their most suitable fitness goal. Possible labels: fat_loss, muscle_gain, maintain. Return only the label.", 3.2, {"instr_adhere": 3, "accuracy": 1, "clarity": 4, "structure": 5, "match": 3}),
    ("HuggingFaceTB/SmolLM2-360M-Instruct", "Ultra-Light Models", 55, 0.03, "classification", "Based on the selected user's UserProfile (weight_kg, height_cm, body_fat_pct, activity_level, goal), determine whether their stated goal is appropriate for their current physical condition. Possible labels: match, mismatch. Return only the label.", 0, {"instr_adhere": 0, "accuracy": 0, "clarity": 0, "structure": 0}),
    ("HuggingFaceTB/SmolLM2-360M-Instruct", "Ultra-Light Models", 77, 0.19, "classification", "Based on the selected user's MealLog records, classify their eating habit by examining which meal_type entries (breakfast, lunch, dinner) appear or are missing across dates. Possible labels: regular_three_meals, skip_breakfast, skip_lunch, skip_dinner, irregular_meals. Return only the label.", 4.5, {"instr_adhere": 4, "accuracy": 4, "clarity": 5, "structure": 5}),
    ("HuggingFaceTB/SmolLM2-360M-Instruct", "Ultra-Light Models", 73, 0.72, "classification", "Based on the selected user's MealLog records, classify their food preference by examining the meal_name entries. Possible labels: asian_only, western_only, salad_heavy, mixed. Return only the label.", 3.5, {"instr_adhere": 4, "accuracy": 2, "clarity": 4, "structure": 4}),
    ("HuggingFaceTB/SmolLM2-360M-Instruct", "Ultra-Light Models", 51, 0.03, "classification", "Based on the selected user's MealLog records, classify their app engagement level by counting meals logged per day and the number of distinct dates. Possible labels: low_engagement, moderate_engagement, high_engagement. Return only the label.", 0, {"instr_adhere": 0, "accuracy": 0, "clarity": 0, "structure": 0, "match": 0}),
    ("HuggingFaceTB/SmolLM2-360M-Instruct", "Ultra-Light Models", 48, 0.03, "prediction", "Based on the selected user's WeightLog history, BodyFatLog history, and UserProfile (goal, activity_level), predict their weight trend over the next 30 days. Return a short prediction with expected direction and estimated change.", 0, {"instr_adhere": 0, "accuracy": 0, "clarity": 0, "structure": 0, "match": 0}),
    ("HuggingFaceTB/SmolLM2-360M-Instruct", "Ultra-Light Models", 59, 0.03, "prediction", "Based on the selected user's MealLog (calories, protein_g, carbs_g, fat_g per meal) and UserProfile (goal, activity_level), predict whether their current intake pattern will support their goal over the next 30 days. Return a short prediction.", 0, {"instr_adhere": 0, "accuracy": 0, "clarity": 0, "structure": 0, "match": 0}),
    ("HuggingFaceTB/SmolLM2-360M-Instruct", "Ultra-Light Models", 146, 2.32, "prediction", "Based on the selected user's MealLog records, predict whether this user is likely to continue logging meals next week, based on the frequency and recency of their entries. Return: likely or unlikely, with a one-line reason.", 3, {"instr_adhere": 3, "accuracy": 3, "clarity": 4, "structure": 2}),
    ("HuggingFaceTB/SmolLM2-360M-Instruct", "Ultra-Light Models", 83, 0.7, "prediction", "Based on the selected user's WeightLog trend, BodyFatLog trend, and UserProfile (goal, activity_level), predict whether they are at risk of not achieving their fitness goal. Return: at_risk or on_track, with a one-line reason.", 3.25, {"instr_adhere": 3, "accuracy": 4, "clarity": 4, "structure": 2}),
    ("HuggingFaceTB/SmolLM2-360M-Instruct", "Ultra-Light Models", 41, 0.03, "prediction", "Based on the selected user's MealLog records, predict whether their eating habit is likely to change in the next month, based on patterns in meal_type and meal_name. Return a short prediction.", 0, {"instr_adhere": 0, "accuracy": 0, "clarity": 0, "structure": 0}),
    ("HuggingFaceTB/SmolLM2-360M-Instruct", "Ultra-Light Models", 122, 2.3, "structured_extraction", "From the selected user's UserNote, extract structured information about their eating preferences. Return as key-value pairs.", 0, {"instr_adhere": 0, "clarity": 0, "structure": 0}),
    ("HuggingFaceTB/SmolLM2-360M-Instruct", "Ultra-Light Models", 69, 1.04, "structured_extraction", "From the selected user's UserNote, extract structured information about their activity level and exercise habits. Return as key-value pairs.", 0, {"instr_adhere": 0, "clarity": 0, "structure": 0}),
    ("HuggingFaceTB/SmolLM2-360M-Instruct", "Ultra-Light Models", 80, 1.18, "structured_extraction", "From the selected user's UserNote, extract structured information about their fitness goal, including target amount and timeline if mentioned. Return as key-value pairs.", 1.3333, {"instr_adhere": 1, "clarity": 1, "structure": 2}),
    ("HuggingFaceTB/SmolLM2-360M-Instruct", "Ultra-Light Models", 120, 2.3, "structured_extraction", "From the selected user's UserNote, extract any dietary restrictions mentioned. Return as key-value pairs.", 0, {"instr_adhere": 0, "clarity": 0, "structure": 0}),
    ("HuggingFaceTB/SmolLM2-360M-Instruct", "Ultra-Light Models", 108, 1.89, "structured_extraction", "From the selected user's UserNote, extract body condition information that may affect their exercise or diet plan. Return as key-value pairs.", 0, {"instr_adhere": 0, "clarity": 0, "structure": 0}),
    ("HuggingFaceTB/SmolLM2-360M-Instruct", "Ultra-Light Models", 129, 2.32, "summarization", "Based on the selected user's MealLog records, summarize their overall eating behavior in one sentence, covering meal timing, calorie intake, and nutritional balance.", 1.3333, {"instr_adhere": 1, "clarity": 1, "structure": 2}),
    ("HuggingFaceTB/SmolLM2-360M-Instruct", "Ultra-Light Models", 49, 0.03, "summarization", "Based on the selected user's UserProfile (weight_kg, height_cm, body_fat_pct, activity_level, goal), provide a one-line summary of whether their stated goal is realistic given their current body metrics.", 1.3333, {"instr_adhere": 1, "clarity": 1, "structure": 2}),
    ("HuggingFaceTB/SmolLM2-360M-Instruct", "Ultra-Light Models", 131, 2.32, "summarization", "Based on the selected user's MealLog records, summarize their diet pattern in one sentence, focusing on meal frequency, meal types, and any noticeable nutritional trends.", 4, {"instr_adhere": 4, "clarity": 4, "structure": 4}),
    ("HuggingFaceTB/SmolLM2-360M-Instruct", "Ultra-Light Models", 125, 2.32, "summarization", "Based on the selected user's MealLog records and BodyFatLog history, summarize their level of progress and consistency in one sentence.", 4, {"instr_adhere": 4, "clarity": 4, "structure": 4}),
    ("HuggingFaceTB/SmolLM2-360M-Instruct", "Ultra-Light Models", 135, 2.32, "summarization", "Based on the selected user's WeightLog history, BodyFatLog history, and UserProfile (goal, activity_level), summarize their overall progress toward their fitness goal in one sentence.", 4, {"instr_adhere": 4, "clarity": 4, "structure": 4}),
    ("HuggingFaceTB/SmolLM2-360M-Instruct", "Ultra-Light Models", 73, 1.34, "text_generation", "Generate a personalised good morning greeting for a user who is on a fat loss goal.", 4, {"instr_adhere": 4, "clarity": 4, "structure": 4}),
    ("HuggingFaceTB/SmolLM2-360M-Instruct", "Ultra-Light Models", 123, 2.3, "text_generation", "Generate a short motivational message for a user who has been consistently logging meals but has not seen weight change in 2 weeks.", 2, {"instr_adhere": 2, "clarity": 2, "structure": 2}),
    ("HuggingFaceTB/SmolLM2-360M-Instruct", "Ultra-Light Models", 115, 2.31, "text_generation", "Generate one practical healthy eating tip suitable for a university student with a busy schedule.", 2.3333, {"instr_adhere": 2, "clarity": 2, "structure": 3}),
    ("HuggingFaceTB/SmolLM2-360M-Instruct", "Ultra-Light Models", 117, 2.3, "text_generation", "Generate a short reminder message for a user who has not logged any meals in the past 3 days.", 3, {"instr_adhere": 3, "clarity": 3, "structure": 3}),
    ("HuggingFaceTB/SmolLM2-360M-Instruct", "Ultra-Light Models", 31, 0.39, "text_generation", "Generate one follow-up question to better understand why a user keeps skipping breakfast.", 4, {"instr_adhere": 4, "clarity": 4, "structure": 4, "tone": 4}),
    ("EleutherAI/pythia-410m", "Ultra-Light Models", 153, 1.07, "classification", "Based on the selected user's UserProfile (height_cm, weight_kg, body_fat_pct, age, activity_level), classify their most suitable fitness goal. Possible labels: fat_loss, muscle_gain, maintain. Return only the label.", 0, {"instr_adhere": 0, "accuracy": 0, "clarity": 0, "structure": 0}),
    ("EleutherAI/pythia-410m", "Ultra-Light Models", 153, 0.96, "classification", "Based on the selected user's UserProfile (weight_kg, height_cm, body_fat_pct, activity_level, goal), determine whether their stated goal is appropriate for their current physical condition. Possible labels: match, mismatch. Return only the label.", 0, {"instr_adhere": 0, "accuracy": 0, "clarity": 0, "structure": 0}),
    ("EleutherAI/pythia-410m", "Ultra-Light Models", 168, 0.96, "classification", "Based on the selected user's MealLog records, classify their eating habit by examining which meal_type entries (breakfast, lunch, dinner) appear or are missing across dates. Possible labels: regular_three_meals, skip_breakfast, skip_lunch, skip_dinner, irregular_meals. Return only the label.", 0, {"instr_adhere": 0, "accuracy": 0, "clarity": 0, "structure": 0}),
    ("EleutherAI/pythia-410m", "Ultra-Light Models", 144, 0.96, "classification", "Based on the selected user's MealLog records, classify their food preference by examining the meal_name entries. Possible labels: asian_only, western_only, salad_heavy, mixed. Return only the label.", 0, {"instr_adhere": 0, "accuracy": 0, "clarity": 0, "structure": 0}),
    ("EleutherAI/pythia-410m", "Ultra-Light Models", 150, 0.96, "classification", "Based on the selected user's MealLog records, classify their app engagement level by counting meals logged per day and the number of distinct dates. Possible labels: low_engagement, moderate_engagement, high_engagement. Return only the label.", 0, {"instr_adhere": 0, "accuracy": 0, "clarity": 0, "structure": 0, "match": 0, "reasonable": 0}),
    ("EleutherAI/pythia-410m", "Ultra-Light Models", 144, 0.96, "prediction", "Based on the selected user's WeightLog history, BodyFatLog history, and UserProfile (goal, activity_level), predict their weight trend over the next 30 days. Return a short prediction with expected direction and estimated change.", 0, {"instr_adhere": 0, "accuracy": 0, "clarity": 0, "structure": 0, "match": 0, "reasonable": 0}),
    ("EleutherAI/pythia-410m", "Ultra-Light Models", 157, 0.96, "prediction", "Based on the selected user's MealLog (calories, protein_g, carbs_g, fat_g per meal) and UserProfile (goal, activity_level), predict whether their current intake pattern will support their goal over the next 30 days. Return a short prediction.", 4, {"instr_adhere": 4, "accuracy": 4, "clarity": 4, "structure": 4}),
    ("EleutherAI/pythia-410m", "Ultra-Light Models", 146, 0.96, "prediction", "Based on the selected user's MealLog records, predict whether this user is likely to continue logging meals next week, based on the frequency and recency of their entries. Return: likely or unlikely, with a one-line reason.", 3.25, {"instr_adhere": 4, "accuracy": 3, "clarity": 4, "structure": 2}),
    ("EleutherAI/pythia-410m", "Ultra-Light Models", 153, 0.96, "prediction", "Based on the selected user's WeightLog trend, BodyFatLog trend, and UserProfile (goal, activity_level), predict whether they are at risk of not achieving their fitness goal. Return: at_risk or on_track, with a one-line reason.", 0, {"instr_adhere": 0, "accuracy": 0, "clarity": 0, "structure": 0}),
    ("EleutherAI/pythia-410m", "Ultra-Light Models", 140, 0.96, "prediction", "Based on the selected user's MealLog records, predict whether their eating habit is likely to change in the next month, based on patterns in meal_type and meal_name. Return a short prediction.", 0, {"instr_adhere": 0, "accuracy": 0, "clarity": 0, "structure": 0}),
    ("EleutherAI/pythia-410m", "Ultra-Light Models", 121, 0.95, "structured_extraction", "From the selected user's UserNote, extract structured information about their eating preferences. Return as key-value pairs.", 2.6667, {"instr_adhere": 2, "clarity": 3, "structure": 3}),
    ("EleutherAI/pythia-410m", "Ultra-Light Models", 124, 0.95, "structured_extraction", "From the selected user's UserNote, extract structured information about their activity level and exercise habits. Return as key-value pairs.", 0, {"instr_adhere": 0, "clarity": 0, "structure": 0}),
    ("EleutherAI/pythia-410m", "Ultra-Light Models", 129, 0.95, "structured_extraction", "From the selected user's UserNote, extract structured information about their fitness goal, including target amount and timeline if mentioned. Return as key-value pairs.", 0, {"instr_adhere": 0, "clarity": 0, "structure": 0}),
    ("EleutherAI/pythia-410m", "Ultra-Light Models", 119, 0.95, "structured_extraction", "From the selected user's UserNote, extract any dietary restrictions mentioned. Return as key-value pairs.", 0, {"instr_adhere": 0, "clarity": 0, "structure": 0}),
    ("EleutherAI/pythia-410m", "Ultra-Light Models", 126, 0.95, "structured_extraction", "From the selected user's UserNote, extract body condition information that may affect their exercise or diet plan. Return as key-value pairs.", 1.3333, {"instr_adhere": 1, "clarity": 2, "structure": 1}),
    ("EleutherAI/pythia-410m", "Ultra-Light Models", 129, 0.96, "summarization", "Based on the selected user's MealLog records, summarize their overall eating behavior in one sentence, covering meal timing, calorie intake, and nutritional balance.", 0, {"instr_adhere": 0, "clarity": 0, "structure": 0, "tone": 0}),
    ("EleutherAI/pythia-410m", "Ultra-Light Models", 145, 0.96, "summarization", "Based on the selected user's UserProfile (weight_kg, height_cm, body_fat_pct, activity_level, goal), provide a one-line summary of whether their stated goal is realistic given their current body metrics.", 0, {"instr_adhere": 0, "clarity": 0, "structure": 0, "tone": 0}),
    ("EleutherAI/pythia-410m", "Ultra-Light Models", 131, 0.96, "summarization", "Based on the selected user's MealLog records, summarize their diet pattern in one sentence, focusing on meal frequency, meal types, and any noticeable nutritional trends.", 2.25, {"instr_adhere": 2, "clarity": 2, "structure": 2, "tone": 3}),
    ("EleutherAI/pythia-410m", "Ultra-Light Models", 125, 0.96, "summarization", "Based on the selected user's MealLog records and BodyFatLog history, summarize their level of progress and consistency in one sentence.", 0.25, {"instr_adhere": 0, "clarity": 0, "structure": 0, "tone": 1}),
    ("EleutherAI/pythia-410m", "Ultra-Light Models", 135, 0.96, "summarization", "Based on the selected user's WeightLog history, BodyFatLog history, and UserProfile (goal, activity_level), summarize their overall progress toward their fitness goal in one sentence.", 0.5, {"instr_adhere": 0, "clarity": 0, "structure": 0, "tone": 2}),
    ("EleutherAI/pythia-410m", "Ultra-Light Models", 116, 0.95, "text_generation", "Generate a personalised good morning greeting for a user who is on a fat loss goal.", 0, {"instr_adhere": 0, "clarity": 0, "structure": 0, "tone": 0}),
    ("EleutherAI/pythia-410m", "Ultra-Light Models", 122, 0.95, "text_generation", "Generate a short motivational message for a user who has been consistently logging meals but has not seen weight change in 2 weeks.", 0.5, {"instr_adhere": 0, "clarity": 1, "structure": 1, "tone": 0}),
    ("EleutherAI/pythia-410m", "Ultra-Light Models", 114, 0.95, "text_generation", "Generate one practical healthy eating tip suitable for a university student with a busy schedule.", 0, {"instr_adhere": 0, "clarity": 0, "structure": 0, "tone": 0}),
    ("EleutherAI/pythia-410m", "Ultra-Light Models", 118, 0.95, "text_generation", "Generate a short reminder message for a user who has not logged any meals in the past 3 days.", 2, {"instr_adhere": 2, "clarity": 2, "structure": 2, "tone": 2}),
    ("EleutherAI/pythia-410m", "Ultra-Light Models", 114, 0.95, "text_generation", "Generate one follow-up question to better understand why a user keeps skipping breakfast.", 0, {"instr_adhere": 0, "clarity": 0, "structure": 0, "tone": 0}),
    ("TinyLlama/TinyLlama-1.1B-Chat-v1.0", "Small Models", 68, 0.1, "classification", "Based on the selected user's UserProfile (height_cm, weight_kg, body_fat_pct, age, activity_level), classify their most suitable fitness goal. Possible labels: fat_loss, muscle_gain, maintain. Return only the label.", 3.75, {"instr_adhere": 3, "accuracy": 3, "clarity": 4, "structure": 5}),
    ("TinyLlama/TinyLlama-1.1B-Chat-v1.0", "Small Models", 101, 0.51, "classification", "Based on the selected user's UserProfile (weight_kg, height_cm, body_fat_pct, activity_level, goal), determine whether their stated goal is appropriate for their current physical condition. Possible labels: match, mismatch. Return only the label.", 3.3333, {"instr_adhere": 4, "accuracy": 2, "clarity": 4, "structure": 4, "match": 4, "reasonable": 2}),
    ("TinyLlama/TinyLlama-1.1B-Chat-v1.0", "Small Models", 118, 0.47, "classification", "Based on the selected user's MealLog records, classify their eating habit by examining which meal_type entries (breakfast, lunch, dinner) appear or are missing across dates. Possible labels: regular_three_meals, skip_breakfast, skip_lunch, skip_dinner, irregular_meals. Return only the label.", 0.25, {"instr_adhere": 1, "accuracy": 0, "clarity": 0, "structure": 0}),
    ("TinyLlama/TinyLlama-1.1B-Chat-v1.0", "Small Models", 153, 1.13, "classification", "Based on the selected user's MealLog records, classify their food preference by examining the meal_name entries. Possible labels: asian_only, western_only, salad_heavy, mixed. Return only the label.", 2.1667, {"instr_adhere": 3, "accuracy": 2, "clarity": 3, "structure": 3, "match": 1, "reasonable": 1}),
    ("TinyLlama/TinyLlama-1.1B-Chat-v1.0", "Small Models", 131, 0.84, "classification", "Based on the selected user's MealLog records, classify their app engagement level by counting meals logged per day and the number of distinct dates. Possible labels: low_engagement, moderate_engagement, high_engagement. Return only the label.", 3, {"instr_adhere": 3, "accuracy": 3, "clarity": 3, "structure": 3, "match": 3, "reasonable": 3}),
    ("TinyLlama/TinyLlama-1.1B-Chat-v1.0", "Small Models", 133, 0.93, "prediction", "Based on the selected user's WeightLog history, BodyFatLog history, and UserProfile (goal, activity_level), predict their weight trend over the next 30 days. Return a short prediction with expected direction and estimated change.", 3, {"instr_adhere": 4, "accuracy": 4, "clarity": 4, "structure": 4, "match": 1, "reasonable": 1}),
    ("TinyLlama/TinyLlama-1.1B-Chat-v1.0", "Small Models", 142, 0.88, "prediction", "Based on the selected user's MealLog (calories, protein_g, carbs_g, fat_g per meal) and UserProfile (goal, activity_level), predict whether their current intake pattern will support their goal over the next 30 days. Return a short prediction.", 2.8333, {"instr_adhere": 3, "accuracy": 3, "clarity": 3, "structure": 3, "match": 3, "reasonable": 2}),
    ("TinyLlama/TinyLlama-1.1B-Chat-v1.0", "Small Models", 74, 0.28, "prediction", "Based on the selected user's MealLog records, predict whether this user is likely to continue logging meals next week, based on the frequency and recency of their entries. Return: likely or unlikely, with a one-line reason.", 3.3333, {"instr_adhere": 4, "accuracy": 3, "clarity": 4, "structure": 4, "match": 3, "reasonable": 2}),
    ("TinyLlama/TinyLlama-1.1B-Chat-v1.0", "Small Models", 102, 0.45, "prediction", "Based on the selected user's WeightLog trend, BodyFatLog trend, and UserProfile (goal, activity_level), predict whether they are at risk of not achieving their fitness goal. Return: at_risk or on_track, with a one-line reason.", 3.5, {"instr_adhere": 4, "accuracy": 4, "clarity": 4, "structure": 4, "match": 3, "reasonable": 2}),
    ("TinyLlama/TinyLlama-1.1B-Chat-v1.0", "Small Models", 125, 0.9, "prediction", "Based on the selected user's MealLog records, predict whether their eating habit is likely to change in the next month, based on patterns in meal_type and meal_name. Return a short prediction.", 3.6667, {"instr_adhere": 4, "accuracy": 4, "clarity": 4, "structure": 4, "match": 3, "reasonable": 3}),
    ("TinyLlama/TinyLlama-1.1B-Chat-v1.0", "Small Models", 105, 0.87, "structured_extraction", "From the selected user's UserNote, extract structured information about their eating preferences. Return as key-value pairs.", 2.3333, {"instr_adhere": 3, "clarity": 3, "structure": 1}),
    ("TinyLlama/TinyLlama-1.1B-Chat-v1.0", "Small Models", 130, 1.1, "structured_extraction", "From the selected user's UserNote, extract structured information about their activity level and exercise habits. Return as key-value pairs.", 0, {"instr_adhere": 0, "clarity": 0, "structure": 0}),
    ("TinyLlama/TinyLlama-1.1B-Chat-v1.0", "Small Models", 136, 1.1, "structured_extraction", "From the selected user's UserNote, extract structured information about their fitness goal, including target amount and timeline if mentioned. Return as key-value pairs.", 1, {"instr_adhere": 1, "clarity": 1, "structure": 1}),
    ("TinyLlama/TinyLlama-1.1B-Chat-v1.0", "Small Models", 86, 0.68, "structured_extraction", "From the selected user's UserNote, extract any dietary restrictions mentioned. Return as key-value pairs.", 0, {"instr_adhere": 0, "clarity": 0, "structure": 0}),
    ("TinyLlama/TinyLlama-1.1B-Chat-v1.0", "Small Models", 76, 0.51, "structured_extraction", "From the selected user's UserNote, extract body condition information that may affect their exercise or diet plan. Return as key-value pairs.", 1.3333, {"instr_adhere": 1, "clarity": 2, "structure": 1}),
    ("TinyLlama/TinyLlama-1.1B-Chat-v1.0", "Small Models", 140, 1.13, "summarization", "Based on the selected user's MealLog records, summarize their overall eating behavior in one sentence, covering meal timing, calorie intake, and nutritional balance.", 4, {"instr_adhere": 4, "clarity": 4, "structure": 4, "tone": 4}),
    ("TinyLlama/TinyLlama-1.1B-Chat-v1.0", "Small Models", 142, 1.03, "summarization", "Based on the selected user's UserProfile (weight_kg, height_cm, body_fat_pct, activity_level, goal), provide a one-line summary of whether their stated goal is realistic given their current body metrics.", 4, {"instr_adhere": 4, "clarity": 4, "structure": 4, "tone": 4}),
    ("TinyLlama/TinyLlama-1.1B-Chat-v1.0", "Small Models", 105, 0.71, "summarization", "Based on the selected user's MealLog records, summarize their diet pattern in one sentence, focusing on meal frequency, meal types, and any noticeable nutritional trends.", 3.25, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 4}),
    ("TinyLlama/TinyLlama-1.1B-Chat-v1.0", "Small Models", 83, 0.61, "summarization", "Based on the selected user's MealLog records and BodyFatLog history, summarize their level of progress and consistency in one sentence.", 2.5, {"instr_adhere": 2, "clarity": 3, "structure": 2, "tone": 3}),
    ("TinyLlama/TinyLlama-1.1B-Chat-v1.0", "Small Models", 102, 0.69, "summarization", "Based on the selected user's WeightLog history, BodyFatLog history, and UserProfile (goal, activity_level), summarize their overall progress toward their fitness goal in one sentence.", 2.25, {"instr_adhere": 3, "clarity": 2, "structure": 1, "tone": 3}),
    ("TinyLlama/TinyLlama-1.1B-Chat-v1.0", "Small Models", 121, 1.1, "text_generation", "Generate a personalised good morning greeting for a user who is on a fat loss goal.", 2.75, {"instr_adhere": 3, "clarity": 2, "structure": 3, "tone": 3}),
    ("TinyLlama/TinyLlama-1.1B-Chat-v1.0", "Small Models", 130, 1.1, "text_generation", "Generate a short motivational message for a user who has been consistently logging meals but has not seen weight change in 2 weeks.", 2.75, {"instr_adhere": 3, "clarity": 2, "structure": 3, "tone": 3}),
    ("TinyLlama/TinyLlama-1.1B-Chat-v1.0", "Small Models", 121, 1.1, "text_generation", "Generate one practical healthy eating tip suitable for a university student with a busy schedule.", 2.5, {"instr_adhere": 2, "clarity": 2, "structure": 3, "tone": 3}),
    ("TinyLlama/TinyLlama-1.1B-Chat-v1.0", "Small Models", 125, 1.1, "text_generation", "Generate a short reminder message for a user who has not logged any meals in the past 3 days.", 2, {"instr_adhere": 2, "clarity": 2, "structure": 2, "tone": 2}),
    ("TinyLlama/TinyLlama-1.1B-Chat-v1.0", "Small Models", 71, 0.59, "text_generation", "Generate one follow-up question to better understand why a user keeps skipping breakfast.", 3, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 3}),
    ("stabilityai/stablelm-2-zephyr-1_6b", "Small Models", 50, 0.04, "classification", "Based on the selected user's UserProfile (height_cm, weight_kg, body_fat_pct, age, activity_level), classify their most suitable fitness goal. Possible labels: fat_loss, muscle_gain, maintain. Return only the label.", 3.75, {"instr_adhere": 3, "accuracy": 3, "clarity": 4, "structure": 5}),
    ("stabilityai/stablelm-2-zephyr-1_6b", "Small Models", 151, 1.06, "classification", "Based on the selected user's UserProfile (weight_kg, height_cm, body_fat_pct, activity_level, goal), determine whether their stated goal is appropriate for their current physical condition. Possible labels: match, mismatch. Return only the label.", 3, {"instr_adhere": 4, "accuracy": 1, "clarity": 4, "structure": 4, "match": 4, "reasonable": 1}),
    ("stabilityai/stablelm-2-zephyr-1_6b", "Small Models", 138, 0.82, "classification", "Based on the selected user's MealLog records, classify their eating habit by examining which meal_type entries (breakfast, lunch, dinner) appear or are missing across dates. Possible labels: regular_three_meals, skip_breakfast, skip_lunch, skip_dinner, irregular_meals. Return only the label.", 2.3333, {"instr_adhere": 3, "accuracy": 1, "clarity": 2, "structure": 2, "match": 3, "reasonable": 3}),
    ("stabilityai/stablelm-2-zephyr-1_6b", "Small Models", 89, 0.55, "classification", "Based on the selected user's MealLog records, classify their food preference by examining the meal_name entries. Possible labels: asian_only, western_only, salad_heavy, mixed. Return only the label.", 3.1667, {"instr_adhere": 3, "accuracy": 3, "clarity": 3, "structure": 3, "match": 4, "reasonable": 3}),
    ("stabilityai/stablelm-2-zephyr-1_6b", "Small Models", 51, 0.07, "classification", "Based on the selected user's MealLog records, classify their app engagement level by counting meals logged per day and the number of distinct dates. Possible labels: low_engagement, moderate_engagement, high_engagement. Return only the label.", 4.75, {"instr_adhere": 5, "accuracy": 4, "clarity": 5, "structure": 5}),
    ("stabilityai/stablelm-2-zephyr-1_6b", "Small Models", 100, 0.59, "prediction", "Based on the selected user's WeightLog history, BodyFatLog history, and UserProfile (goal, activity_level), predict their weight trend over the next 30 days. Return a short prediction with expected direction and estimated change.", 4, {"instr_adhere": 4, "accuracy": 4, "clarity": 4, "structure": 4}),
    ("stabilityai/stablelm-2-zephyr-1_6b", "Small Models", 108, 0.58, "prediction", "Based on the selected user's MealLog (calories, protein_g, carbs_g, fat_g per meal) and UserProfile (goal, activity_level), predict whether their current intake pattern will support their goal over the next 30 days. Return a short prediction.", 2.3333, {"instr_adhere": 3, "accuracy": 3, "clarity": 3, "structure": 3, "match": 1, "reasonable": 1}),
    ("stabilityai/stablelm-2-zephyr-1_6b", "Small Models", 126, 0.85, "prediction", "Based on the selected user's MealLog records, predict whether this user is likely to continue logging meals next week, based on the frequency and recency of their entries. Return: likely or unlikely, with a one-line reason.", 3.5, {"instr_adhere": 4, "accuracy": 3, "clarity": 4, "structure": 4, "match": 3, "reasonable": 3}),
    ("stabilityai/stablelm-2-zephyr-1_6b", "Small Models", 151, 1.06, "prediction", "Based on the selected user's WeightLog trend, BodyFatLog trend, and UserProfile (goal, activity_level), predict whether they are at risk of not achieving their fitness goal. Return: at_risk or on_track, with a one-line reason.", 3.6667, {"instr_adhere": 4, "accuracy": 4, "clarity": 4, "structure": 4, "match": 3, "reasonable": 3}),
    ("stabilityai/stablelm-2-zephyr-1_6b", "Small Models", 86, 0.51, "prediction", "Based on the selected user's MealLog records, predict whether their eating habit is likely to change in the next month, based on patterns in meal_type and meal_name. Return a short prediction.", 3.6667, {"instr_adhere": 4, "accuracy": 4, "clarity": 4, "structure": 4, "match": 3, "reasonable": 3}),
    ("stabilityai/stablelm-2-zephyr-1_6b", "Small Models", 52, 0.33, "structured_extraction", "From the selected user's UserNote, extract structured information about their eating preferences. Return as key-value pairs.", 3.6667, {"instr_adhere": 3, "clarity": 4, "structure": 4}),
    ("stabilityai/stablelm-2-zephyr-1_6b", "Small Models", 73, 0.52, "structured_extraction", "From the selected user's UserNote, extract structured information about their activity level and exercise habits. Return as key-value pairs.", 3, {"instr_adhere": 3, "clarity": 4, "structure": 2}),
    ("stabilityai/stablelm-2-zephyr-1_6b", "Small Models", 129, 1.03, "structured_extraction", "From the selected user's UserNote, extract structured information about their fitness goal, including target amount and timeline if mentioned. Return as key-value pairs.", 2.3333, {"instr_adhere": 2, "clarity": 1, "structure": 4}),
    ("stabilityai/stablelm-2-zephyr-1_6b", "Small Models", 54, 0.37, "structured_extraction", "From the selected user's UserNote, extract any dietary restrictions mentioned. Return as key-value pairs.", 1, {"instr_adhere": 1, "clarity": 1, "structure": 1}),
    ("stabilityai/stablelm-2-zephyr-1_6b", "Small Models", 120, 0.98, "structured_extraction", "From the selected user's UserNote, extract body condition information that may affect their exercise or diet plan. Return as key-value pairs.", 3, {"instr_adhere": 3, "clarity": 3, "structure": 3}),
    ("stabilityai/stablelm-2-zephyr-1_6b", "Small Models", 94, 0.69, "summarization", "Based on the selected user's MealLog records, summarize their overall eating behavior in one sentence, covering meal timing, calorie intake, and nutritional balance.", 4, {"instr_adhere": 4, "clarity": 4, "structure": 4, "tone": 4}),
    ("stabilityai/stablelm-2-zephyr-1_6b", "Small Models", 84, 0.45, "summarization", "Based on the selected user's UserProfile (weight_kg, height_cm, body_fat_pct, activity_level, goal), provide a one-line summary of whether their stated goal is realistic given their current body metrics.", 4, {"instr_adhere": 4, "clarity": 4, "structure": 4, "tone": 4}),
    ("stabilityai/stablelm-2-zephyr-1_6b", "Small Models", 85, 0.58, "summarization", "Based on the selected user's MealLog records, summarize their diet pattern in one sentence, focusing on meal frequency, meal types, and any noticeable nutritional trends.", 3.25, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 4}),
    ("stabilityai/stablelm-2-zephyr-1_6b", "Small Models", 52, 0.3, "summarization", "Based on the selected user's MealLog records and BodyFatLog history, summarize their level of progress and consistency in one sentence.", 3, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 3}),
    ("stabilityai/stablelm-2-zephyr-1_6b", "Small Models", 66, 0.35, "summarization", "Based on the selected user's WeightLog history, BodyFatLog history, and UserProfile (goal, activity_level), summarize their overall progress toward their fitness goal in one sentence.", 3, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 3}),
    ("stabilityai/stablelm-2-zephyr-1_6b", "Small Models", 94, 0.82, "text_generation", "Generate a personalised good morning greeting for a user who is on a fat loss goal.", 3.25, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 4}),
    ("stabilityai/stablelm-2-zephyr-1_6b", "Small Models", 124, 1.03, "text_generation", "Generate a short motivational message for a user who has been consistently logging meals but has not seen weight change in 2 weeks.", 2.75, {"instr_adhere": 3, "clarity": 2, "structure": 3, "tone": 3}),
    ("stabilityai/stablelm-2-zephyr-1_6b", "Small Models", 115, 1.03, "text_generation", "Generate one practical healthy eating tip suitable for a university student with a busy schedule.", 3.75, {"instr_adhere": 4, "clarity": 4, "structure": 4, "tone": 3}),
    ("stabilityai/stablelm-2-zephyr-1_6b", "Small Models", 70, 0.52, "text_generation", "Generate a short reminder message for a user who has not logged any meals in the past 3 days.", 4, {"instr_adhere": 4, "clarity": 4, "structure": 4, "tone": 4}),
    ("stabilityai/stablelm-2-zephyr-1_6b", "Small Models", 82, 0.7, "text_generation", "Generate one follow-up question to better understand why a user keeps skipping breakfast.", 3.25, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 4}),
    ("microsoft/phi-2", "Small Models", 155, 1.69, "classification", "Based on the selected user's UserProfile (height_cm, weight_kg, body_fat_pct, age, activity_level), classify their most suitable fitness goal. Possible labels: fat_loss, muscle_gain, maintain. Return only the label.", 2.25, {"instr_adhere": 1, "accuracy": 4, "clarity": 2, "structure": 2}),
    ("microsoft/phi-2", "Small Models", 155, 1.59, "classification", "Based on the selected user's UserProfile (weight_kg, height_cm, body_fat_pct, activity_level, goal), determine whether their stated goal is appropriate for their current physical condition. Possible labels: match, mismatch. Return only the label.", 3.3333, {"instr_adhere": 4, "accuracy": 2, "clarity": 4, "structure": 4, "match": 4, "reasonable": 2}),
    ("microsoft/phi-2", "Small Models", 169, 1.57, "classification", "Based on the selected user's MealLog records, classify their eating habit by examining which meal_type entries (breakfast, lunch, dinner) appear or are missing across dates. Possible labels: regular_three_meals, skip_breakfast, skip_lunch, skip_dinner, irregular_meals. Return only the label.", 3.5, {"instr_adhere": 3, "accuracy": 4, "clarity": 3, "structure": 3, "match": 4, "reasonable": 4}),
    ("microsoft/phi-2", "Small Models", 146, 1.57, "classification", "Based on the selected user's MealLog records, classify their food preference by examining the meal_name entries. Possible labels: asian_only, western_only, salad_heavy, mixed. Return only the label.", 3.5, {"instr_adhere": 4, "accuracy": 4, "clarity": 4, "structure": 4, "match": 3, "reasonable": 2}),
    ("microsoft/phi-2", "Small Models", 149, 1.56, "classification", "Based on the selected user's MealLog records, classify their app engagement level by counting meals logged per day and the number of distinct dates. Possible labels: low_engagement, moderate_engagement, high_engagement. Return only the label.", 3.5, {"instr_adhere": 4, "accuracy": 3, "clarity": 4, "structure": 4, "match": 3, "reasonable": 3}),
    ("microsoft/phi-2", "Small Models", 114, 1.13, "prediction", "Based on the selected user's WeightLog history, BodyFatLog history, and UserProfile (goal, activity_level), predict their weight trend over the next 30 days. Return a short prediction with expected direction and estimated change.", 3.1667, {"instr_adhere": 4, "accuracy": 4, "clarity": 4, "structure": 4, "match": 2, "reasonable": 1}),
    ("microsoft/phi-2", "Small Models", 157, 1.57, "prediction", "Based on the selected user's MealLog (calories, protein_g, carbs_g, fat_g per meal) and UserProfile (goal, activity_level), predict whether their current intake pattern will support their goal over the next 30 days. Return a short prediction.", 2.3333, {"instr_adhere": 3, "accuracy": 3, "clarity": 3, "structure": 3, "match": 1, "reasonable": 1}),
    ("microsoft/phi-2", "Small Models", 148, 1.57, "prediction", "Based on the selected user's MealLog records, predict whether this user is likely to continue logging meals next week, based on the frequency and recency of their entries. Return: likely or unlikely, with a one-line reason.", 3.1667, {"instr_adhere": 4, "accuracy": 3, "clarity": 4, "structure": 4, "match": 2, "reasonable": 2}),
    ("microsoft/phi-2", "Small Models", 153, 1.59, "prediction", "Based on the selected user's WeightLog trend, BodyFatLog trend, and UserProfile (goal, activity_level), predict whether they are at risk of not achieving their fitness goal. Return: at_risk or on_track, with a one-line reason.", 3, {"instr_adhere": 3, "accuracy": 3, "clarity": 3, "structure": 3, "match": 3, "reasonable": 3}),
    ("microsoft/phi-2", "Small Models", 120, 1.28, "prediction", "Based on the selected user's MealLog records, predict whether their eating habit is likely to change in the next month, based on patterns in meal_type and meal_name. Return a short prediction.", 3.6667, {"instr_adhere": 4, "accuracy": 4, "clarity": 4, "structure": 4, "match": 3, "reasonable": 3}),
    ("microsoft/phi-2", "Small Models", 101, 1.42, "structured_extraction", "From the selected user's UserNote, extract structured information about their eating preferences. Return as key-value pairs.", 1.6, {"instr_adhere": 3, "clarity": 1, "structure": 1, "match": 2, "reasonable": 1}),
    ("microsoft/phi-2", "Small Models", 121, 1.54, "structured_extraction", "From the selected user's UserNote, extract structured information about their activity level and exercise habits. Return as key-value pairs.", 2, {"instr_adhere": 2, "clarity": 2, "structure": 2}),
    ("microsoft/phi-2", "Small Models", 118, 1.52, "structured_extraction", "From the selected user's UserNote, extract structured information about their fitness goal, including target amount and timeline if mentioned. Return as key-value pairs.", 0.6667, {"instr_adhere": 0, "clarity": 1, "structure": 1}),
    ("microsoft/phi-2", "Small Models", 115, 1.54, "structured_extraction", "From the selected user's UserNote, extract any dietary restrictions mentioned. Return as key-value pairs.", 1, {"instr_adhere": 1, "clarity": 1, "structure": 1}),
    ("microsoft/phi-2", "Small Models", 95, 1.07, "structured_extraction", "From the selected user's UserNote, extract body condition information that may affect their exercise or diet plan. Return as key-value pairs.", 2, {"instr_adhere": 2, "clarity": 2, "structure": 2}),
    ("microsoft/phi-2", "Small Models", 109, 1.28, "summarization", "Based on the selected user's MealLog records, summarize their overall eating behavior in one sentence, covering meal timing, calorie intake, and nutritional balance.", 2.5, {"instr_adhere": 2, "clarity": 3, "structure": 2, "tone": 3}),
    ("microsoft/phi-2", "Small Models", 149, 1.57, "summarization", "Based on the selected user's UserProfile (weight_kg, height_cm, body_fat_pct, activity_level, goal), provide a one-line summary of whether their stated goal is realistic given their current body metrics.", 3.5, {"instr_adhere": 3, "clarity": 4, "structure": 3, "tone": 4}),
    ("microsoft/phi-2", "Small Models", 116, 1.36, "summarization", "Based on the selected user's MealLog records, summarize their diet pattern in one sentence, focusing on meal frequency, meal types, and any noticeable nutritional trends.", 3.25, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 4}),
    ("microsoft/phi-2", "Small Models", 126, 1.57, "summarization", "Based on the selected user's MealLog records and BodyFatLog history, summarize their level of progress and consistency in one sentence.", 2.75, {"instr_adhere": 3, "clarity": 3, "structure": 2, "tone": 3}),
    ("microsoft/phi-2", "Small Models", 137, 1.57, "summarization", "Based on the selected user's WeightLog history, BodyFatLog history, and UserProfile (goal, activity_level), summarize their overall progress toward their fitness goal in one sentence.", 2.75, {"instr_adhere": 3, "clarity": 3, "structure": 2, "tone": 3}),
    ("microsoft/phi-2", "Small Models", 119, 1.53, "text_generation", "Generate a personalised good morning greeting for a user who is on a fat loss goal.", 2.25, {"instr_adhere": 2, "clarity": 2, "structure": 2, "tone": 3}),
    ("microsoft/phi-2", "Small Models", 124, 1.53, "text_generation", "Generate a short motivational message for a user who has been consistently logging meals but has not seen weight change in 2 weeks.", 2.25, {"instr_adhere": 2, "clarity": 2, "structure": 2, "tone": 3}),
    ("microsoft/phi-2", "Small Models", 43, 0.41, "text_generation", "Generate one practical healthy eating tip suitable for a university student with a busy schedule.", 3, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 3}),
    ("microsoft/phi-2", "Small Models", 53, 0.54, "text_generation", "Generate a short reminder message for a user who has not logged any meals in the past 3 days.", 4, {"instr_adhere": 4, "clarity": 4, "structure": 4, "tone": 4}),
    ("microsoft/phi-2", "Small Models", 115, 1.53, "text_generation", "Generate one follow-up question to better understand why a user keeps skipping breakfast.", 2, {"instr_adhere": 2, "clarity": 2, "structure": 2, "tone": 2}),
    ("Qwen/Qwen2.5-3B-Instruct", "Medium Models", 148, 2.05, "classification", "Based on the selected user's UserProfile (height_cm, weight_kg, body_fat_pct, age, activity_level), classify their most suitable fitness goal. Possible labels: fat_loss, muscle_gain, maintain. Return only the label.", 1.8333, {"instr_adhere": 2, "accuracy": 1, "clarity": 1, "structure": 2, "match": 3, "reasonable": 2}),
    ("Qwen/Qwen2.5-3B-Instruct", "Medium Models", 151, 1.98, "classification", "Based on the selected user's UserProfile (weight_kg, height_cm, body_fat_pct, activity_level, goal), determine whether their stated goal is appropriate for their current physical condition. Possible labels: match, mismatch. Return only the label.", 3.6667, {"instr_adhere": 4, "accuracy": 3, "clarity": 4, "structure": 4, "match": 4, "reasonable": 3}),
    ("Qwen/Qwen2.5-3B-Instruct", "Medium Models", 164, 1.98, "classification", "Based on the selected user's MealLog records, classify their eating habit by examining which meal_type entries (breakfast, lunch, dinner) appear or are missing across dates. Possible labels: regular_three_meals, skip_breakfast, skip_lunch, skip_dinner, irregular_meals. Return only the label.", 3.6667, {"instr_adhere": 4, "accuracy": 4, "clarity": 4, "structure": 4, "match": 3, "reasonable": 3}),
    ("Qwen/Qwen2.5-3B-Instruct", "Medium Models", 142, 1.98, "classification", "Based on the selected user's MealLog records, classify their food preference by examining the meal_name entries. Possible labels: asian_only, western_only, salad_heavy, mixed. Return only the label.", 3.3333, {"instr_adhere": 4, "accuracy": 3, "clarity": 4, "structure": 4, "match": 3, "reasonable": 2}),
    ("Qwen/Qwen2.5-3B-Instruct", "Medium Models", 149, 1.98, "classification", "Based on the selected user's MealLog records, classify their app engagement level by counting meals logged per day and the number of distinct dates. Possible labels: low_engagement, moderate_engagement, high_engagement. Return only the label.", 3.75, {"instr_adhere": 4, "accuracy": 3, "clarity": 4, "structure": 4}),
    ("Qwen/Qwen2.5-3B-Instruct", "Medium Models", 146, 1.98, "prediction", "Based on the selected user's WeightLog history, BodyFatLog history, and UserProfile (goal, activity_level), predict their weight trend over the next 30 days. Return a short prediction with expected direction and estimated change.", 3.5, {"instr_adhere": 4, "accuracy": 4, "clarity": 4, "structure": 4, "match": 3, "reasonable": 2}),
    ("Qwen/Qwen2.5-3B-Instruct", "Medium Models", 154, 1.98, "prediction", "Based on the selected user's MealLog (calories, protein_g, carbs_g, fat_g per meal) and UserProfile (goal, activity_level), predict whether their current intake pattern will support their goal over the next 30 days. Return a short prediction.", 3, {"instr_adhere": 3, "accuracy": 3, "clarity": 3, "structure": 3, "match": 3, "reasonable": 3}),
    ("Qwen/Qwen2.5-3B-Instruct", "Medium Models", 146, 1.98, "prediction", "Based on the selected user's MealLog records, predict whether this user is likely to continue logging meals next week, based on the frequency and recency of their entries. Return: likely or unlikely, with a one-line reason.", 3.3333, {"instr_adhere": 4, "accuracy": 3, "clarity": 4, "structure": 4, "match": 2, "reasonable": 3}),
    ("Qwen/Qwen2.5-3B-Instruct", "Medium Models", 151, 1.98, "prediction", "Based on the selected user's WeightLog trend, BodyFatLog trend, and UserProfile (goal, activity_level), predict whether they are at risk of not achieving their fitness goal. Return: at_risk or on_track, with a one-line reason.", 3.6667, {"instr_adhere": 4, "accuracy": 4, "clarity": 4, "structure": 4, "match": 3, "reasonable": 3}),
    ("Qwen/Qwen2.5-3B-Instruct", "Medium Models", 139, 1.98, "prediction", "Based on the selected user's MealLog records, predict whether their eating habit is likely to change in the next month, based on patterns in meal_type and meal_name. Return a short prediction.", 3.6667, {"instr_adhere": 4, "accuracy": 4, "clarity": 4, "structure": 4, "match": 3, "reasonable": 3}),
    ("Qwen/Qwen2.5-3B-Instruct", "Medium Models", 122, 1.98, "structured_extraction", "From the selected user's UserNote, extract structured information about their eating preferences. Return as key-value pairs.", 3, {"instr_adhere": 3, "clarity": 3, "structure": 3}),
    ("Qwen/Qwen2.5-3B-Instruct", "Medium Models", 125, 1.94, "structured_extraction", "From the selected user's UserNote, extract structured information about their activity level and exercise habits. Return as key-value pairs.", 3, {"instr_adhere": 3, "clarity": 3, "structure": 3}),
    ("Qwen/Qwen2.5-3B-Instruct", "Medium Models", 130, 1.94, "structured_extraction", "From the selected user's UserNote, extract structured information about their fitness goal, including target amount and timeline if mentioned. Return as key-value pairs.", 0, {"instr_adhere": 0, "clarity": 0, "structure": 0}),
    ("Qwen/Qwen2.5-3B-Instruct", "Medium Models", 105, 1.66, "structured_extraction", "From the selected user's UserNote, extract any dietary restrictions mentioned. Return as key-value pairs.", 2.6667, {"instr_adhere": 3, "clarity": 3, "structure": 2}),
    ("Qwen/Qwen2.5-3B-Instruct", "Medium Models", 127, 1.94, "structured_extraction", "From the selected user's UserNote, extract body condition information that may affect their exercise or diet plan. Return as key-value pairs.", 1.3333, {"instr_adhere": 1, "clarity": 2, "structure": 1}),
    ("Qwen/Qwen2.5-3B-Instruct", "Medium Models", 130, 1.98, "summarization", "Based on the selected user's MealLog records, summarize their overall eating behavior in one sentence, covering meal timing, calorie intake, and nutritional balance.", 2.5, {"instr_adhere": 2, "clarity": 3, "structure": 2, "tone": 3}),
    ("Qwen/Qwen2.5-3B-Instruct", "Medium Models", 143, 1.98, "summarization", "Based on the selected user's UserProfile (weight_kg, height_cm, body_fat_pct, activity_level, goal), provide a one-line summary of whether their stated goal is realistic given their current body metrics.", 2.75, {"instr_adhere": 3, "clarity": 3, "structure": 2, "tone": 3}),
    ("Qwen/Qwen2.5-3B-Instruct", "Medium Models", 132, 1.99, "summarization", "Based on the selected user's MealLog records, summarize their diet pattern in one sentence, focusing on meal frequency, meal types, and any noticeable nutritional trends.", 3, {"instr_adhere": 3, "clarity": 3, "structure": 2, "tone": 4}),
    ("Qwen/Qwen2.5-3B-Instruct", "Medium Models", 126, 1.98, "summarization", "Based on the selected user's MealLog records and BodyFatLog history, summarize their level of progress and consistency in one sentence.", 2.75, {"instr_adhere": 3, "clarity": 3, "structure": 2, "tone": 3}),
    ("Qwen/Qwen2.5-3B-Instruct", "Medium Models", 135, 1.98, "summarization", "Based on the selected user's WeightLog history, BodyFatLog history, and UserProfile (goal, activity_level), summarize their overall progress toward their fitness goal in one sentence.", 3, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 3}),
    ("Qwen/Qwen2.5-3B-Instruct", "Medium Models", 117, 1.93, "text_generation", "Generate a personalised good morning greeting for a user who is on a fat loss goal.", 2.25, {"instr_adhere": 2, "clarity": 2, "structure": 2, "tone": 3}),
    ("Qwen/Qwen2.5-3B-Instruct", "Medium Models", 125, 1.93, "text_generation", "Generate a short motivational message for a user who has been consistently logging meals but has not seen weight change in 2 weeks.", 2.75, {"instr_adhere": 3, "clarity": 2, "structure": 3, "tone": 3}),
    ("Qwen/Qwen2.5-3B-Instruct", "Medium Models", 116, 1.93, "text_generation", "Generate one practical healthy eating tip suitable for a university student with a busy schedule.", 3.5, {"instr_adhere": 3, "clarity": 4, "structure": 4, "tone": 3}),
    ("Qwen/Qwen2.5-3B-Instruct", "Medium Models", 121, 1.93, "text_generation", "Generate a short reminder message for a user who has not logged any meals in the past 3 days.", 3.25, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 4}),
    ("Qwen/Qwen2.5-3B-Instruct", "Medium Models", 115, 1.93, "text_generation", "Generate one follow-up question to better understand why a user keeps skipping breakfast.", 3.25, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 4}),
    ("microsoft/Phi-3-mini-4k-instruct", "Medium Models", 161, 1.79, "classification", "Based on the selected user's UserProfile (height_cm, weight_kg, body_fat_pct, age, activity_level), classify their most suitable fitness goal. Possible labels: fat_loss, muscle_gain, maintain. Return only the label.", 3.1667, {"instr_adhere": 3, "accuracy": 4, "clarity": 4, "structure": 5, "match": 2, "reasonable": 1}),
    ("microsoft/Phi-3-mini-4k-instruct", "Medium Models", 157, 1.78, "classification", "Based on the selected user's UserProfile (weight_kg, height_cm, body_fat_pct, activity_level, goal), determine whether their stated goal is appropriate for their current physical condition. Possible labels: match, mismatch. Return only the label.", 3, {"instr_adhere": 4, "accuracy": 4, "clarity": 4, "structure": 4, "match": 1, "reasonable": 1}),
    ("microsoft/Phi-3-mini-4k-instruct", "Medium Models", 178, 1.8, "classification", "Based on the selected user's MealLog records, classify their eating habit by examining which meal_type entries (breakfast, lunch, dinner) appear or are missing across dates. Possible labels: regular_three_meals, skip_breakfast, skip_lunch, skip_dinner, irregular_meals. Return only the label.", 4, {"instr_adhere": 4, "accuracy": 4, "clarity": 4, "structure": 4}),
    ("microsoft/Phi-3-mini-4k-instruct", "Medium Models", 151, 1.86, "classification", "Based on the selected user's MealLog records, classify their food preference by examining the meal_name entries. Possible labels: asian_only, western_only, salad_heavy, mixed. Return only the label.", 4, {"instr_adhere": 4, "accuracy": 4, "clarity": 4, "structure": 4}),
    ("microsoft/Phi-3-mini-4k-instruct", "Medium Models", 156, 1.86, "classification", "Based on the selected user's MealLog records, classify their app engagement level by counting meals logged per day and the number of distinct dates. Possible labels: low_engagement, moderate_engagement, high_engagement. Return only the label.", 3.5, {"instr_adhere": 4, "accuracy": 2, "clarity": 4, "structure": 4}),
    ("microsoft/Phi-3-mini-4k-instruct", "Medium Models", 150, 1.86, "prediction", "Based on the selected user's WeightLog history, BodyFatLog history, and UserProfile (goal, activity_level), predict their weight trend over the next 30 days. Return a short prediction with expected direction and estimated change.", 3.5, {"instr_adhere": 4, "accuracy": 4, "clarity": 4, "structure": 4, "match": 3, "reasonable": 2}),
    ("microsoft/Phi-3-mini-4k-instruct", "Medium Models", 163, 1.87, "prediction", "Based on the selected user's MealLog (calories, protein_g, carbs_g, fat_g per meal) and UserProfile (goal, activity_level), predict whether their current intake pattern will support their goal over the next 30 days. Return a short prediction.", 2.3333, {"instr_adhere": 2, "accuracy": 2, "clarity": 2, "structure": 2, "match": 3, "reasonable": 3}),
    ("microsoft/Phi-3-mini-4k-instruct", "Medium Models", 150, 1.78, "prediction", "Based on the selected user's MealLog records, predict whether this user is likely to continue logging meals next week, based on the frequency and recency of their entries. Return: likely or unlikely, with a one-line reason.", 3.3333, {"instr_adhere": 4, "accuracy": 3, "clarity": 4, "structure": 3, "match": 3, "reasonable": 3}),
    ("microsoft/Phi-3-mini-4k-instruct", "Medium Models", 162, 1.87, "prediction", "Based on the selected user's WeightLog trend, BodyFatLog trend, and UserProfile (goal, activity_level), predict whether they are at risk of not achieving their fitness goal. Return: at_risk or on_track, with a one-line reason.", 4, {"instr_adhere": 4, "accuracy": 4, "clarity": 4, "structure": 4}),
    ("microsoft/Phi-3-mini-4k-instruct", "Medium Models", 144, 1.78, "prediction", "Based on the selected user's MealLog records, predict whether their eating habit is likely to change in the next month, based on patterns in meal_type and meal_name. Return a short prediction.", 3.6667, {"instr_adhere": 4, "accuracy": 4, "clarity": 4, "structure": 4, "match": 3, "reasonable": 3}),
    ("microsoft/Phi-3-mini-4k-instruct", "Medium Models", 126, 1.68, "structured_extraction", "From the selected user's UserNote, extract structured information about their eating preferences. Return as key-value pairs.", 2.3333, {"instr_adhere": 1, "clarity": 3, "structure": 3}),
    ("microsoft/Phi-3-mini-4k-instruct", "Medium Models", 128, 1.68, "structured_extraction", "From the selected user's UserNote, extract structured information about their activity level and exercise habits. Return as key-value pairs.", 3.3333, {"instr_adhere": 3, "clarity": 3, "structure": 4}),
    ("microsoft/Phi-3-mini-4k-instruct", "Medium Models", 134, 1.68, "structured_extraction", "From the selected user's UserNote, extract structured information about their fitness goal, including target amount and timeline if mentioned. Return as key-value pairs.", 4, {"instr_adhere": 4, "clarity": 4, "structure": 4}),
    ("microsoft/Phi-3-mini-4k-instruct", "Medium Models", 123, 1.68, "structured_extraction", "From the selected user's UserNote, extract any dietary restrictions mentioned. Return as key-value pairs.", 0.6667, {"instr_adhere": 0, "clarity": 1, "structure": 1}),
    ("microsoft/Phi-3-mini-4k-instruct", "Medium Models", 129, 1.69, "structured_extraction", "From the selected user's UserNote, extract body condition information that may affect their exercise or diet plan. Return as key-value pairs.", 3.3333, {"instr_adhere": 4, "clarity": 3, "structure": 3}),
    ("microsoft/Phi-3-mini-4k-instruct", "Medium Models", 138, 1.78, "summarization", "Based on the selected user's MealLog records, summarize their overall eating behavior in one sentence, covering meal timing, calorie intake, and nutritional balance.", 4, {"instr_adhere": 4, "clarity": 4, "structure": 4, "tone": 4}),
    ("microsoft/Phi-3-mini-4k-instruct", "Medium Models", 150, 1.78, "summarization", "Based on the selected user's UserProfile (weight_kg, height_cm, body_fat_pct, activity_level, goal), provide a one-line summary of whether their stated goal is realistic given their current body metrics.", 4, {"instr_adhere": 4, "clarity": 4, "structure": 4, "tone": 4}),
    ("microsoft/Phi-3-mini-4k-instruct", "Medium Models", 142, 1.78, "summarization", "Based on the selected user's MealLog records, summarize their diet pattern in one sentence, focusing on meal frequency, meal types, and any noticeable nutritional trends.", 2.25, {"instr_adhere": 2, "clarity": 2, "structure": 2, "tone": 3}),
    ("microsoft/Phi-3-mini-4k-instruct", "Medium Models", 130, 1.78, "summarization", "Based on the selected user's MealLog records and BodyFatLog history, summarize their level of progress and consistency in one sentence.", 2.75, {"instr_adhere": 3, "clarity": 3, "structure": 2, "tone": 3}),
    ("microsoft/Phi-3-mini-4k-instruct", "Medium Models", 141, 1.78, "summarization", "Based on the selected user's WeightLog history, BodyFatLog history, and UserProfile (goal, activity_level), summarize their overall progress toward their fitness goal in one sentence.", 2.75, {"instr_adhere": 3, "clarity": 3, "structure": 2, "tone": 3}),
    ("microsoft/Phi-3-mini-4k-instruct", "Medium Models", 120, 1.67, "text_generation", "Generate a personalised good morning greeting for a user who is on a fat loss goal.", 2.25, {"instr_adhere": 2, "clarity": 2, "structure": 2, "tone": 3}),
    ("microsoft/Phi-3-mini-4k-instruct", "Medium Models", 128, 1.68, "text_generation", "Generate a short motivational message for a user who has been consistently logging meals but has not seen weight change in 2 weeks.", 3, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 3}),
    ("microsoft/Phi-3-mini-4k-instruct", "Medium Models", 118, 1.68, "text_generation", "Generate one practical healthy eating tip suitable for a university student with a busy schedule.", 3.5, {"instr_adhere": 3, "clarity": 4, "structure": 4, "tone": 3}),
    ("microsoft/Phi-3-mini-4k-instruct", "Medium Models", 123, 1.68, "text_generation", "Generate a short reminder message for a user who has not logged any meals in the past 3 days.", 3.25, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 4}),
    ("microsoft/Phi-3-mini-4k-instruct", "Medium Models", 117, 1.67, "text_generation", "Generate one follow-up question to better understand why a user keeps skipping breakfast.", 3, {"instr_adhere": 3, "clarity": 3, "structure": 2, "tone": 4}),
    ("stabilityai/stablelm-zephyr-3b", "Medium Models", 58, 0.09, "classification", "Based on the selected user's UserProfile (height_cm, weight_kg, body_fat_pct, age, activity_level), classify their most suitable fitness goal. Possible labels: fat_loss, muscle_gain, maintain. Return only the label.", 3.25, {"instr_adhere": 3, "accuracy": 1, "clarity": 4, "structure": 5}),
    ("stabilityai/stablelm-zephyr-3b", "Medium Models", 155, 1.38, "classification", "Based on the selected user's UserProfile (weight_kg, height_cm, body_fat_pct, activity_level, goal), determine whether their stated goal is appropriate for their current physical condition. Possible labels: match, mismatch. Return only the label.", 3.6667, {"instr_adhere": 4, "accuracy": 3, "clarity": 4, "structure": 4, "match": 4, "reasonable": 3}),
    ("stabilityai/stablelm-zephyr-3b", "Medium Models", 171, 1.38, "classification", "Based on the selected user's MealLog records, classify their eating habit by examining which meal_type entries (breakfast, lunch, dinner) appear or are missing across dates. Possible labels: regular_three_meals, skip_breakfast, skip_lunch, skip_dinner, irregular_meals. Return only the label.", 3.8333, {"instr_adhere": 3, "accuracy": 4, "clarity": 4, "structure": 4, "match": 4, "reasonable": 4}),
    ("stabilityai/stablelm-zephyr-3b", "Medium Models", 50, 0.09, "classification", "Based on the selected user's MealLog records, classify their food preference by examining the meal_name entries. Possible labels: asian_only, western_only, salad_heavy, mixed. Return only the label.", 4.5, {"instr_adhere": 5, "accuracy": 3, "clarity": 5, "structure": 5}),
    ("stabilityai/stablelm-zephyr-3b", "Medium Models", 73, 0.34, "classification", "Based on the selected user's MealLog records, classify their app engagement level by counting meals logged per day and the number of distinct dates. Possible labels: low_engagement, moderate_engagement, high_engagement. Return only the label.", 3, {"instr_adhere": 3, "accuracy": 3, "clarity": 3, "structure": 3}),
    ("stabilityai/stablelm-zephyr-3b", "Medium Models", 138, 1.3, "prediction", "Based on the selected user's WeightLog history, BodyFatLog history, and UserProfile (goal, activity_level), predict their weight trend over the next 30 days. Return a short prediction with expected direction and estimated change.", 3.5, {"instr_adhere": 4, "accuracy": 4, "clarity": 4, "structure": 4, "match": 3, "reasonable": 2}),
    ("stabilityai/stablelm-zephyr-3b", "Medium Models", 152, 1.32, "prediction", "Based on the selected user's MealLog (calories, protein_g, carbs_g, fat_g per meal) and UserProfile (goal, activity_level), predict whether their current intake pattern will support their goal over the next 30 days. Return a short prediction.", 3, {"instr_adhere": 3, "accuracy": 3, "clarity": 3, "structure": 3, "match": 3, "reasonable": 3}),
    ("stabilityai/stablelm-zephyr-3b", "Medium Models", 137, 1.26, "prediction", "Based on the selected user's MealLog records, predict whether this user is likely to continue logging meals next week, based on the frequency and recency of their entries. Return: likely or unlikely, with a one-line reason.", 3, {"instr_adhere": 3, "accuracy": 3, "clarity": 3, "structure": 3, "match": 3, "reasonable": 3}),
    ("stabilityai/stablelm-zephyr-3b", "Medium Models", 58, 0.09, "prediction", "Based on the selected user's WeightLog trend, BodyFatLog trend, and UserProfile (goal, activity_level), predict whether they are at risk of not achieving their fitness goal. Return: at_risk or on_track, with a one-line reason.", 3.6667, {"instr_adhere": 4, "accuracy": 4, "clarity": 4, "structure": 4, "match": 3, "reasonable": 3}),
    ("stabilityai/stablelm-zephyr-3b", "Medium Models", 131, 1.26, "prediction", "Based on the selected user's MealLog records, predict whether their eating habit is likely to change in the next month, based on patterns in meal_type and meal_name. Return a short prediction.", 3.6667, {"instr_adhere": 4, "accuracy": 4, "clarity": 4, "structure": 4, "match": 3, "reasonable": 3}),
    ("stabilityai/stablelm-zephyr-3b", "Medium Models", 122, 1.28, "structured_extraction", "From the selected user's UserNote, extract structured information about their eating preferences. Return as key-value pairs.", 1.3333, {"instr_adhere": 2, "clarity": 1, "structure": 1}),
    ("stabilityai/stablelm-zephyr-3b", "Medium Models", 125, 1.28, "structured_extraction", "From the selected user's UserNote, extract structured information about their activity level and exercise habits. Return as key-value pairs.", 3.6667, {"instr_adhere": 3, "clarity": 4, "structure": 4}),
    ("stabilityai/stablelm-zephyr-3b", "Medium Models", 69, 0.51, "structured_extraction", "From the selected user's UserNote, extract structured information about their fitness goal, including target amount and timeline if mentioned. Return as key-value pairs.", 4, {"instr_adhere": 4, "clarity": 4, "structure": 4}),
    ("stabilityai/stablelm-zephyr-3b", "Medium Models", 39, 0.26, "structured_extraction", "From the selected user's UserNote, extract any dietary restrictions mentioned. Return as key-value pairs.", 4, {"instr_adhere": 4, "clarity": 4, "structure": 4}),
    ("stabilityai/stablelm-zephyr-3b", "Medium Models", 87, 0.78, "structured_extraction", "From the selected user's UserNote, extract body condition information that may affect their exercise or diet plan. Return as key-value pairs.", 3, {"instr_adhere": 3, "clarity": 3, "structure": 3}),
    ("stabilityai/stablelm-zephyr-3b", "Medium Models", 130, 1.42, "summarization", "Based on the selected user's MealLog records, summarize their overall eating behavior in one sentence, covering meal timing, calorie intake, and nutritional balance.", 4, {"instr_adhere": 4, "clarity": 4, "structure": 4, "tone": 4}),
    ("stabilityai/stablelm-zephyr-3b", "Medium Models", 114, 0.94, "summarization", "Based on the selected user's UserProfile (weight_kg, height_cm, body_fat_pct, activity_level, goal), provide a one-line summary of whether their stated goal is realistic given their current body metrics.", 4, {"instr_adhere": 4, "clarity": 4, "structure": 4, "tone": 4}),
    ("stabilityai/stablelm-zephyr-3b", "Medium Models", 100, 0.96, "summarization", "Based on the selected user's MealLog records, summarize their diet pattern in one sentence, focusing on meal frequency, meal types, and any noticeable nutritional trends.", 3.25, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 4}),
    ("stabilityai/stablelm-zephyr-3b", "Medium Models", 92, 0.94, "summarization", "Based on the selected user's MealLog records and BodyFatLog history, summarize their level of progress and consistency in one sentence.", 3, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 3}),
    ("stabilityai/stablelm-zephyr-3b", "Medium Models", 85, 0.71, "summarization", "Based on the selected user's WeightLog history, BodyFatLog history, and UserProfile (goal, activity_level), summarize their overall progress toward their fitness goal in one sentence.", 3.75, {"instr_adhere": 4, "clarity": 4, "structure": 4, "reasonable": "4+N222", "tone": 3}),
    ("stabilityai/stablelm-zephyr-3b", "Medium Models", 78, 0.78, "text_generation", "Generate a personalised good morning greeting for a user who is on a fat loss goal.", 3.25, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 4}),
    ("stabilityai/stablelm-zephyr-3b", "Medium Models", 65, 0.54, "text_generation", "Generate a short motivational message for a user who has been consistently logging meals but has not seen weight change in 2 weeks.", 3.75, {"instr_adhere": 3, "clarity": 4, "structure": 4, "tone": 4}),
    ("stabilityai/stablelm-zephyr-3b", "Medium Models", 114, 1.26, "text_generation", "Generate one practical healthy eating tip suitable for a university student with a busy schedule.", 3.5, {"instr_adhere": 3, "clarity": 4, "structure": 4, "tone": 3}),
    ("stabilityai/stablelm-zephyr-3b", "Medium Models", 58, 0.52, "text_generation", "Generate a short reminder message for a user who has not logged any meals in the past 3 days.", 4, {"instr_adhere": 4, "clarity": 4, "structure": 4, "tone": 4}),
    ("stabilityai/stablelm-zephyr-3b", "Medium Models", 91, 0.96, "text_generation", "Generate one follow-up question to better understand why a user keeps skipping breakfast.", 3, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 3}),
    ("HuggingFaceH4/zephyr-7b-beta", "Large Models", 140, 2.22, "classification", "Based on the selected user's UserProfile (height_cm, weight_kg, body_fat_pct, age, activity_level), classify their most suitable fitness goal. Possible labels: fat_loss, muscle_gain, maintain. Return only the label.", 2.5, {"instr_adhere": 2, "accuracy": 3, "clarity": 3, "structure": 3, "match": 3, "reasonable": 1}),
    ("HuggingFaceH4/zephyr-7b-beta", "Large Models", 62, 0.17, "classification", "Based on the selected user's UserProfile (weight_kg, height_cm, body_fat_pct, activity_level, goal), determine whether their stated goal is appropriate for their current physical condition. Possible labels: match, mismatch. Return only the label.", 3.25, {"instr_adhere": 4, "accuracy": 1, "clarity": 4, "structure": 4}),
    ("HuggingFaceH4/zephyr-7b-beta", "Large Models", 135, 1.66, "classification", "Based on the selected user's MealLog records, classify their eating habit by examining which meal_type entries (breakfast, lunch, dinner) appear or are missing across dates. Possible labels: regular_three_meals, skip_breakfast, skip_lunch, skip_dinner, irregular_meals. Return only the label.", 2.5, {"instr_adhere": 2, "accuracy": 1, "clarity": 3, "structure": 3, "match": 4, "reasonable": 2}),
    ("HuggingFaceH4/zephyr-7b-beta", "Large Models", 104, 1.53, "classification", "Based on the selected user's MealLog records, classify their food preference by examining the meal_name entries. Possible labels: asian_only, western_only, salad_heavy, mixed. Return only the label.", 2.8333, {"instr_adhere": 3, "accuracy": 3, "clarity": 3, "structure": 3, "match": 3, "reasonable": 2}),
    ("HuggingFaceH4/zephyr-7b-beta", "Large Models", 153, 2.67, "classification", "Based on the selected user's MealLog records, classify their app engagement level by counting meals logged per day and the number of distinct dates. Possible labels: low_engagement, moderate_engagement, high_engagement. Return only the label.", 3, {"instr_adhere": 3, "accuracy": 3, "clarity": 3, "structure": 3}),
    ("HuggingFaceH4/zephyr-7b-beta", "Large Models", 151, 2.77, "prediction", "Based on the selected user's WeightLog history, BodyFatLog history, and UserProfile (goal, activity_level), predict their weight trend over the next 30 days. Return a short prediction with expected direction and estimated change.", 3.3333, {"instr_adhere": 4, "accuracy": 4, "clarity": 4, "structure": 4, "match": 2, "reasonable": 2}),
    ("HuggingFaceH4/zephyr-7b-beta", "Large Models", 165, 2.77, "prediction", "Based on the selected user's MealLog (calories, protein_g, carbs_g, fat_g per meal) and UserProfile (goal, activity_level), predict whether their current intake pattern will support their goal over the next 30 days. Return a short prediction.", 3, {"instr_adhere": 3, "accuracy": 3, "clarity": 3, "structure": 3, "match": 3, "reasonable": 3}),
    ("HuggingFaceH4/zephyr-7b-beta", "Large Models", 150, 2.72, "prediction", "Based on the selected user's MealLog records, predict whether this user is likely to continue logging meals next week, based on the frequency and recency of their entries. Return: likely or unlikely, with a one-line reason.", 3, {"instr_adhere": 3, "accuracy": 3, "clarity": 3, "structure": 3, "match": 3, "reasonable": 3}),
    ("HuggingFaceH4/zephyr-7b-beta", "Large Models", 159, 2.72, "prediction", "Based on the selected user's WeightLog trend, BodyFatLog trend, and UserProfile (goal, activity_level), predict whether they are at risk of not achieving their fitness goal. Return: at_risk or on_track, with a one-line reason.", 3, {"instr_adhere": 3, "accuracy": 3, "clarity": 3, "structure": 3, "match": 3, "reasonable": 3}),
    ("HuggingFaceH4/zephyr-7b-beta", "Large Models", 144, 2.72, "prediction", "Based on the selected user's MealLog records, predict whether their eating habit is likely to change in the next month, based on patterns in meal_type and meal_name. Return a short prediction.", 3.6667, {"instr_adhere": 4, "accuracy": 4, "clarity": 4, "structure": 4, "match": 3, "reasonable": 3}),
    ("HuggingFaceH4/zephyr-7b-beta", "Large Models", 124, 2.6, "structured_extraction", "From the selected user's UserNote, extract structured information about their eating preferences. Return as key-value pairs.", 3.6667, {"instr_adhere": 3, "clarity": 4, "structure": 4}),
    ("HuggingFaceH4/zephyr-7b-beta", "Large Models", 64, 0.97, "structured_extraction", "From the selected user's UserNote, extract structured information about their activity level and exercise habits. Return as key-value pairs.", 4, {"instr_adhere": 4, "clarity": 4, "structure": 4}),
    ("HuggingFaceH4/zephyr-7b-beta", "Large Models", 134, 2.6, "structured_extraction", "From the selected user's UserNote, extract structured information about their fitness goal, including target amount and timeline if mentioned. Return as key-value pairs.", 4, {"instr_adhere": 4, "clarity": 4, "structure": 4}),
    ("HuggingFaceH4/zephyr-7b-beta", "Large Models", 60, 0.97, "structured_extraction", "From the selected user's UserNote, extract any dietary restrictions mentioned. Return as key-value pairs.", 1, {"instr_adhere": 1, "clarity": 1, "structure": 1}),
    ("HuggingFaceH4/zephyr-7b-beta", "Large Models", 62, 0.86, "structured_extraction", "From the selected user's UserNote, extract body condition information that may affect their exercise or diet plan. Return as key-value pairs.", 3.6667, {"instr_adhere": 4, "clarity": 4, "structure": 3}),
    ("HuggingFaceH4/zephyr-7b-beta", "Large Models", 115, 2.11, "summarization", "Based on the selected user's MealLog records, summarize their overall eating behavior in one sentence, covering meal timing, calorie intake, and nutritional balance.", 4, {"instr_adhere": 4, "clarity": 4, "structure": 4, "tone": 4}),
    ("HuggingFaceH4/zephyr-7b-beta", "Large Models", 152, 2.77, "summarization", "Based on the selected user's UserProfile (weight_kg, height_cm, body_fat_pct, activity_level, goal), provide a one-line summary of whether their stated goal is realistic given their current body metrics.", 4, {"instr_adhere": 4, "clarity": 4, "structure": 4, "tone": 4}),
    ("HuggingFaceH4/zephyr-7b-beta", "Large Models", 140, 2.72, "summarization", "Based on the selected user's MealLog records, summarize their diet pattern in one sentence, focusing on meal frequency, meal types, and any noticeable nutritional trends.", 3.25, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 4}),
    ("HuggingFaceH4/zephyr-7b-beta", "Large Models", 94, 1.77, "summarization", "Based on the selected user's MealLog records and BodyFatLog history, summarize their level of progress and consistency in one sentence.", 3, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 3}),
    ("HuggingFaceH4/zephyr-7b-beta", "Large Models", 107, 1.82, "summarization", "Based on the selected user's WeightLog history, BodyFatLog history, and UserProfile (goal, activity_level), summarize their overall progress toward their fitness goal in one sentence.", 3.75, {"instr_adhere": 4, "clarity": 4, "structure": 4, "tone": 3}),
    ("HuggingFaceH4/zephyr-7b-beta", "Large Models", 90, 1.84, "text_generation", "Generate a personalised good morning greeting for a user who is on a fat loss goal.", 3.25, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 4}),
    ("HuggingFaceH4/zephyr-7b-beta", "Large Models", 126, 2.59, "text_generation", "Generate a short motivational message for a user who has been consistently logging meals but has not seen weight change in 2 weeks.", 3.25, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 4}),
    ("HuggingFaceH4/zephyr-7b-beta", "Large Models", 117, 2.59, "text_generation", "Generate one practical healthy eating tip suitable for a university student with a busy schedule.", 3.5, {"instr_adhere": 3, "clarity": 4, "structure": 4, "tone": 3}),
    ("HuggingFaceH4/zephyr-7b-beta", "Large Models", 98, 2, "text_generation", "Generate a short reminder message for a user who has not logged any meals in the past 3 days.", 3.25, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 4}),
    ("HuggingFaceH4/zephyr-7b-beta", "Large Models", 92, 1.94, "text_generation", "Generate one follow-up question to better understand why a user keeps skipping breakfast.", 2.75, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 2}),
    ("NousResearch/Nous-Hermes-2-Mistral-7B-DPO", "Large Models", 65, 0.25, "classification", "Based on the selected user's UserProfile (height_cm, weight_kg, body_fat_pct, age, activity_level), classify their most suitable fitness goal. Possible labels: fat_loss, muscle_gain, maintain. Return only the label.", 3.75, {"instr_adhere": 3, "accuracy": 3, "clarity": 4, "structure": 5}),
    ("NousResearch/Nous-Hermes-2-Mistral-7B-DPO", "Large Models", 112, 1.48, "classification", "Based on the selected user's UserProfile (weight_kg, height_cm, body_fat_pct, activity_level, goal), determine whether their stated goal is appropriate for their current physical condition. Possible labels: match, mismatch. Return only the label.", 2.8333, {"instr_adhere": 4, "accuracy": 1, "clarity": 4, "structure": 4, "match": 3, "reasonable": 1}),
    ("NousResearch/Nous-Hermes-2-Mistral-7B-DPO", "Large Models", 85, 0.31, "classification", "Based on the selected user's MealLog records, classify their eating habit by examining which meal_type entries (breakfast, lunch, dinner) appear or are missing across dates. Possible labels: regular_three_meals, skip_breakfast, skip_lunch, skip_dinner, irregular_meals. Return only the label.", 4.25, {"instr_adhere": 4, "accuracy": 4, "clarity": 4, "structure": 5}),
    ("NousResearch/Nous-Hermes-2-Mistral-7B-DPO", "Large Models", 55, 0.2, "classification", "Based on the selected user's MealLog records, classify their food preference by examining the meal_name entries. Possible labels: asian_only, western_only, salad_heavy, mixed. Return only the label.", 4.5, {"instr_adhere": 5, "accuracy": 3, "clarity": 5, "structure": 5}),
    ("NousResearch/Nous-Hermes-2-Mistral-7B-DPO", "Large Models", 63, 0.3, "classification", "Based on the selected user's MealLog records, classify their app engagement level by counting meals logged per day and the number of distinct dates. Possible labels: low_engagement, moderate_engagement, high_engagement. Return only the label.", 4.5, {"instr_adhere": 5, "accuracy": 3, "clarity": 5, "structure": 5}),
    ("NousResearch/Nous-Hermes-2-Mistral-7B-DPO", "Large Models", 139, 2.48, "prediction", "Based on the selected user's WeightLog history, BodyFatLog history, and UserProfile (goal, activity_level), predict their weight trend over the next 30 days. Return a short prediction with expected direction and estimated change.", 3.3333, {"instr_adhere": 4, "accuracy": 4, "clarity": 4, "structure": 4, "match": 2, "reasonable": 2}),
    ("NousResearch/Nous-Hermes-2-Mistral-7B-DPO", "Large Models", 165, 2.72, "prediction", "Based on the selected user's MealLog (calories, protein_g, carbs_g, fat_g per meal) and UserProfile (goal, activity_level), predict whether their current intake pattern will support their goal over the next 30 days. Return a short prediction.", 3, {"instr_adhere": 3, "accuracy": 3, "clarity": 3, "structure": 3, "match": 3, "reasonable": 3}),
    ("NousResearch/Nous-Hermes-2-Mistral-7B-DPO", "Large Models", 149, 2.69, "prediction", "Based on the selected user's MealLog records, predict whether this user is likely to continue logging meals next week, based on the frequency and recency of their entries. Return: likely or unlikely, with a one-line reason.", 3.5, {"instr_adhere": 4, "accuracy": 3, "clarity": 4, "structure": 4, "match": 3, "reasonable": 3}),
    ("NousResearch/Nous-Hermes-2-Mistral-7B-DPO", "Large Models", 159, 2.72, "prediction", "Based on the selected user's WeightLog trend, BodyFatLog trend, and UserProfile (goal, activity_level), predict whether they are at risk of not achieving their fitness goal. Return: at_risk or on_track, with a one-line reason.", 3.6667, {"instr_adhere": 4, "accuracy": 4, "clarity": 4, "structure": 4, "match": 3, "reasonable": 3}),
    ("NousResearch/Nous-Hermes-2-Mistral-7B-DPO", "Large Models", 144, 2.72, "prediction", "Based on the selected user's MealLog records, predict whether their eating habit is likely to change in the next month, based on patterns in meal_type and meal_name. Return a short prediction.", 3.6667, {"instr_adhere": 4, "accuracy": 4, "clarity": 4, "structure": 4, "match": 3, "reasonable": 3}),
    ("NousResearch/Nous-Hermes-2-Mistral-7B-DPO", "Large Models", 125, 2.6, "structured_extraction", "From the selected user's UserNote, extract structured information about their eating preferences. Return as key-value pairs.", 2.8, {"instr_adhere": 2, "clarity": 3, "structure": 3, "match": 3, "reasonable": 3}),
    ("NousResearch/Nous-Hermes-2-Mistral-7B-DPO", "Large Models", 121, 2.45, "structured_extraction", "From the selected user's UserNote, extract structured information about their activity level and exercise habits. Return as key-value pairs.", 2.6667, {"instr_adhere": 3, "clarity": 2, "structure": 3}),
    ("NousResearch/Nous-Hermes-2-Mistral-7B-DPO", "Large Models", 130, 2.52, "structured_extraction", "From the selected user's UserNote, extract structured information about their fitness goal, including target amount and timeline if mentioned. Return as key-value pairs.", 4, {"instr_adhere": 4, "clarity": 4, "structure": 4}),
    ("NousResearch/Nous-Hermes-2-Mistral-7B-DPO", "Large Models", 56, 0.86, "structured_extraction", "From the selected user's UserNote, extract any dietary restrictions mentioned. Return as key-value pairs.", 2.6667, {"instr_adhere": 4, "clarity": 2, "structure": 2}),
    ("NousResearch/Nous-Hermes-2-Mistral-7B-DPO", "Large Models", 130, 2.6, "structured_extraction", "From the selected user's UserNote, extract body condition information that may affect their exercise or diet plan. Return as key-value pairs.", 2.3333, {"instr_adhere": 2, "clarity": 3, "structure": 2}),
    ("NousResearch/Nous-Hermes-2-Mistral-7B-DPO", "Large Models", 138, 2.72, "summarization", "Based on the selected user's MealLog records, summarize their overall eating behavior in one sentence, covering meal timing, calorie intake, and nutritional balance.", 4, {"instr_adhere": 4, "clarity": 4, "structure": 4, "tone": 4}),
    ("NousResearch/Nous-Hermes-2-Mistral-7B-DPO", "Large Models", 89, 1.09, "summarization", "Based on the selected user's UserProfile (weight_kg, height_cm, body_fat_pct, activity_level, goal), provide a one-line summary of whether their stated goal is realistic given their current body metrics.", 4, {"instr_adhere": 4, "clarity": 4, "structure": 4, "tone": 4}),
    ("NousResearch/Nous-Hermes-2-Mistral-7B-DPO", "Large Models", 109, 1.96, "summarization", "Based on the selected user's MealLog records, summarize their diet pattern in one sentence, focusing on meal frequency, meal types, and any noticeable nutritional trends.", 3.25, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 4}),
    ("NousResearch/Nous-Hermes-2-Mistral-7B-DPO", "Large Models", 68, 1.12, "summarization", "Based on the selected user's MealLog records and BodyFatLog history, summarize their level of progress and consistency in one sentence.", 3, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 3}),
    ("NousResearch/Nous-Hermes-2-Mistral-7B-DPO", "Large Models", 70, 0.88, "summarization", "Based on the selected user's WeightLog history, BodyFatLog history, and UserProfile (goal, activity_level), summarize their overall progress toward their fitness goal in one sentence.", 3.75, {"instr_adhere": 4, "clarity": 4, "structure": 4, "tone": 3}),
    ("NousResearch/Nous-Hermes-2-Mistral-7B-DPO", "Large Models", 76, 1.48, "text_generation", "Generate a personalised good morning greeting for a user who is on a fat loss goal.", 3, {"instr_adhere": 3, "clarity": 2, "structure": 3, "tone": 4}),
    ("NousResearch/Nous-Hermes-2-Mistral-7B-DPO", "Large Models", 113, 2.26, "text_generation", "Generate a short motivational message for a user who has been consistently logging meals but has not seen weight change in 2 weeks.", 3.75, {"instr_adhere": 3, "clarity": 4, "structure": 4, "tone": 4}),
    ("NousResearch/Nous-Hermes-2-Mistral-7B-DPO", "Large Models", 117, 2.59, "text_generation", "Generate one practical healthy eating tip suitable for a university student with a busy schedule.", 3.5, {"instr_adhere": 3, "clarity": 4, "structure": 4, "tone": 3}),
    ("NousResearch/Nous-Hermes-2-Mistral-7B-DPO", "Large Models", 121, 2.59, "text_generation", "Generate a short reminder message for a user who has not logged any meals in the past 3 days.", 3.25, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 4}),
    ("NousResearch/Nous-Hermes-2-Mistral-7B-DPO", "Large Models", 32, 0.39, "text_generation", "Generate one follow-up question to better understand why a user keeps skipping breakfast.", 3, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 3}),
    ("tiiuae/falcon-7b-instruct", "Large Models", 78, 1.2, "classification", "Based on the selected user's UserProfile (height_cm, weight_kg, body_fat_pct, age, activity_level), classify their most suitable fitness goal. Possible labels: fat_loss, muscle_gain, maintain. Return only the label.", 2.5, {"instr_adhere": 2, "accuracy": 2, "clarity": 3, "structure": 3}),
    ("tiiuae/falcon-7b-instruct", "Large Models", 124, 2.66, "classification", "Based on the selected user's UserProfile (weight_kg, height_cm, body_fat_pct, activity_level, goal), determine whether their stated goal is appropriate for their current physical condition. Possible labels: match, mismatch. Return only the label.", 3.3333, {"instr_adhere": 3, "accuracy": 3, "clarity": 3, "structure": 3, "match": 4, "reasonable": 4}),
    ("tiiuae/falcon-7b-instruct", "Large Models", 99, 1.18, "classification", "Based on the selected user's MealLog records, classify their eating habit by examining which meal_type entries (breakfast, lunch, dinner) appear or are missing across dates. Possible labels: regular_three_meals, skip_breakfast, skip_lunch, skip_dinner, irregular_meals. Return only the label.", 3.5, {"instr_adhere": 3, "accuracy": 3, "clarity": 4, "structure": 4}),
    ("tiiuae/falcon-7b-instruct", "Large Models", 67, 1.03, "classification", "Based on the selected user's MealLog records, classify their food preference by examining the meal_name entries. Possible labels: asian_only, western_only, salad_heavy, mixed. Return only the label.", 2.3333, {"instr_adhere": 3, "accuracy": 3, "clarity": 3, "structure": 3, "match": 1, "reasonable": 1}),
    ("tiiuae/falcon-7b-instruct", "Large Models", 54, 0.47, "classification", "Based on the selected user's MealLog records, classify their app engagement level by counting meals logged per day and the number of distinct dates. Possible labels: low_engagement, moderate_engagement, high_engagement. Return only the label.", 4.25, {"instr_adhere": 5, "accuracy": 2, "clarity": 5, "structure": 5}),
    ("tiiuae/falcon-7b-instruct", "Large Models", 83, 1.46, "prediction", "Based on the selected user's WeightLog history, BodyFatLog history, and UserProfile (goal, activity_level), predict their weight trend over the next 30 days. Return a short prediction with expected direction and estimated change.", 4, {"instr_adhere": 4, "accuracy": 4, "clarity": 4, "structure": 4}),
    ("tiiuae/falcon-7b-instruct", "Large Models", 116, 2.2, "prediction", "Based on the selected user's MealLog (calories, protein_g, carbs_g, fat_g per meal) and UserProfile (goal, activity_level), predict whether their current intake pattern will support their goal over the next 30 days. Return a short prediction.", 3, {"instr_adhere": 3, "accuracy": 3, "clarity": 3, "structure": 3, "match": 3, "reasonable": 3}),
    ("tiiuae/falcon-7b-instruct", "Large Models", 95, 1.85, "prediction", "Based on the selected user's MealLog records, predict whether this user is likely to continue logging meals next week, based on the frequency and recency of their entries. Return: likely or unlikely, with a one-line reason.", 3.5, {"instr_adhere": 4, "accuracy": 3, "clarity": 4, "structure": 4, "match": 3, "reasonable": 3}),
    ("tiiuae/falcon-7b-instruct", "Large Models", 109, 2.06, "prediction", "Based on the selected user's WeightLog trend, BodyFatLog trend, and UserProfile (goal, activity_level), predict whether they are at risk of not achieving their fitness goal. Return: at_risk or on_track, with a one-line reason.", 3.6667, {"instr_adhere": 4, "accuracy": 4, "clarity": 4, "structure": 4, "match": 3, "reasonable": 3}),
    ("tiiuae/falcon-7b-instruct", "Large Models", 119, 2.98, "prediction", "Based on the selected user's MealLog records, predict whether their eating habit is likely to change in the next month, based on patterns in meal_type and meal_name. Return a short prediction.", 3.6667, {"instr_adhere": 4, "accuracy": 4, "clarity": 4, "structure": 4, "match": 3, "reasonable": 3}),
    ("tiiuae/falcon-7b-instruct", "Large Models", 123, 3.55, "structured_extraction", "From the selected user's UserNote, extract structured information about their eating preferences. Return as key-value pairs.", 1.3333, {"instr_adhere": 1, "clarity": 1, "structure": 2}),
    ("tiiuae/falcon-7b-instruct", "Large Models", 126, 3.54, "structured_extraction", "From the selected user's UserNote, extract structured information about their activity level and exercise habits. Return as key-value pairs.", 4, {"instr_adhere": 4, "clarity": 4, "structure": 4}),
    ("tiiuae/falcon-7b-instruct", "Large Models", 131, 3.54, "structured_extraction", "From the selected user's UserNote, extract structured information about their fitness goal, including target amount and timeline if mentioned. Return as key-value pairs.", 1, {"instr_adhere": 1, "clarity": 1, "structure": 1}),
    ("tiiuae/falcon-7b-instruct", "Large Models", 121, 3.55, "structured_extraction", "From the selected user's UserNote, extract any dietary restrictions mentioned. Return as key-value pairs.", 0, {"instr_adhere": 0, "clarity": 0, "structure": 0}),
    ("tiiuae/falcon-7b-instruct", "Large Models", 127, 3.55, "structured_extraction", "From the selected user's UserNote, extract body condition information that may affect their exercise or diet plan. Return as key-value pairs.", 1.3333, {"instr_adhere": 1, "clarity": 2, "structure": 1}),
    ("tiiuae/falcon-7b-instruct", "Large Models", 69, 1.56, "summarization", "Based on the selected user's MealLog records, summarize their overall eating behavior in one sentence, covering meal timing, calorie intake, and nutritional balance.", 4, {"instr_adhere": 4, "clarity": 4, "structure": 4, "tone": 4}),
    ("tiiuae/falcon-7b-instruct", "Large Models", 150, 3.73, "summarization", "Based on the selected user's UserProfile (weight_kg, height_cm, body_fat_pct, activity_level, goal), provide a one-line summary of whether their stated goal is realistic given their current body metrics.", 4, {"instr_adhere": 4, "clarity": 4, "structure": 4, "tone": 4}),
    ("tiiuae/falcon-7b-instruct", "Large Models", 114, 3.08, "summarization", "Based on the selected user's MealLog records, summarize their diet pattern in one sentence, focusing on meal frequency, meal types, and any noticeable nutritional trends.", 3.25, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 4}),
    ("tiiuae/falcon-7b-instruct", "Large Models", 81, 2.16, "summarization", "Based on the selected user's MealLog records and BodyFatLog history, summarize their level of progress and consistency in one sentence.", 3, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 3}),
    ("tiiuae/falcon-7b-instruct", "Large Models", 138, 3.72, "summarization", "Based on the selected user's WeightLog history, BodyFatLog history, and UserProfile (goal, activity_level), summarize their overall progress toward their fitness goal in one sentence.", 3, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 3}),
    ("tiiuae/falcon-7b-instruct", "Large Models", 101, 3.09, "text_generation", "Generate a personalised good morning greeting for a user who is on a fat loss goal.", 3, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 3}),
    ("tiiuae/falcon-7b-instruct", "Large Models", 82, 2.06, "text_generation", "Generate a short motivational message for a user who has been consistently logging meals but has not seen weight change in 2 weeks.", 3.75, {"instr_adhere": 3, "clarity": 4, "structure": 4, "tone": 4}),
    ("tiiuae/falcon-7b-instruct", "Large Models", 56, 1.49, "text_generation", "Generate one practical healthy eating tip suitable for a university student with a busy schedule.", 3.5, {"instr_adhere": 3, "clarity": 4, "structure": 4, "tone": 3}),
    ("tiiuae/falcon-7b-instruct", "Large Models", 57, 1.31, "text_generation", "Generate a short reminder message for a user who has not logged any meals in the past 3 days.", 4, {"instr_adhere": 4, "clarity": 4, "structure": 4, "tone": 4}),
    ("tiiuae/falcon-7b-instruct", "Large Models", 27, 0.43, "text_generation", "Generate one follow-up question to better understand why a user keeps skipping breakfast.", 3, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 3}),
    ("01-ai/Yi-1.5-9B-Chat", "Extra-Large Models", 157, 3.47, "classification", "Based on the selected user's UserProfile (height_cm, weight_kg, body_fat_pct, age, activity_level), classify their most suitable fitness goal. Possible labels: fat_loss, muscle_gain, maintain. Return only the label.", 2.1667, {"instr_adhere": 2, "accuracy": 1, "clarity": 3, "structure": 3, "match": 3, "reasonable": 1}),
    ("01-ai/Yi-1.5-9B-Chat", "Extra-Large Models", 155, 3.47, "classification", "Based on the selected user's UserProfile (weight_kg, height_cm, body_fat_pct, activity_level, goal), determine whether their stated goal is appropriate for their current physical condition. Possible labels: match, mismatch. Return only the label.", 3.1667, {"instr_adhere": 3, "accuracy": 3, "clarity": 3, "structure": 3, "match": 4, "reasonable": 3}),
    ("01-ai/Yi-1.5-9B-Chat", "Extra-Large Models", 173, 3.47, "classification", "Based on the selected user's MealLog records, classify their eating habit by examining which meal_type entries (breakfast, lunch, dinner) appear or are missing across dates. Possible labels: regular_three_meals, skip_breakfast, skip_lunch, skip_dinner, irregular_meals. Return only the label.", 3.6667, {"instr_adhere": 4, "accuracy": 4, "clarity": 4, "structure": 4, "match": 4, "reasonable": 2}),
    ("01-ai/Yi-1.5-9B-Chat", "Extra-Large Models", 143, 3.47, "classification", "Based on the selected user's MealLog records, classify their food preference by examining the meal_name entries. Possible labels: asian_only, western_only, salad_heavy, mixed. Return only the label.", 2.8333, {"instr_adhere": 3, "accuracy": 3, "clarity": 3, "structure": 3, "match": 3, "reasonable": 2}),
    ("01-ai/Yi-1.5-9B-Chat", "Extra-Large Models", 153, 3.47, "classification", "Based on the selected user's MealLog records, classify their app engagement level by counting meals logged per day and the number of distinct dates. Possible labels: low_engagement, moderate_engagement, high_engagement. Return only the label.", 2.1667, {"instr_adhere": 2, "accuracy": 1, "clarity": 2, "structure": 2, "match": 3, "reasonable": 3}),
    ("01-ai/Yi-1.5-9B-Chat", "Extra-Large Models", 150, 3.47, "prediction", "Based on the selected user's WeightLog history, BodyFatLog history, and UserProfile (goal, activity_level), predict their weight trend over the next 30 days. Return a short prediction with expected direction and estimated change.", 3.6667, {"instr_adhere": 4, "accuracy": 4, "clarity": 4, "structure": 4, "match": 3, "reasonable": 3}),
    ("01-ai/Yi-1.5-9B-Chat", "Extra-Large Models", 161, 3.54, "prediction", "Based on the selected user's MealLog (calories, protein_g, carbs_g, fat_g per meal) and UserProfile (goal, activity_level), predict whether their current intake pattern will support their goal over the next 30 days. Return a short prediction.", 3, {"instr_adhere": 3, "accuracy": 3, "clarity": 3, "structure": 3, "match": 3, "reasonable": 3}),
    ("01-ai/Yi-1.5-9B-Chat", "Extra-Large Models", 147, 3.47, "prediction", "Based on the selected user's MealLog records, predict whether this user is likely to continue logging meals next week, based on the frequency and recency of their entries. Return: likely or unlikely, with a one-line reason.", 3.5, {"instr_adhere": 4, "accuracy": 3, "clarity": 4, "structure": 4, "match": 3, "reasonable": 3}),
    ("01-ai/Yi-1.5-9B-Chat", "Extra-Large Models", 157, 3.47, "prediction", "Based on the selected user's WeightLog trend, BodyFatLog trend, and UserProfile (goal, activity_level), predict whether they are at risk of not achieving their fitness goal. Return: at_risk or on_track, with a one-line reason.", 3, {"instr_adhere": 3, "accuracy": 3, "clarity": 3, "structure": 3, "match": 3, "reasonable": 3}),
    ("01-ai/Yi-1.5-9B-Chat", "Extra-Large Models", 142, 3.47, "prediction", "Based on the selected user's MealLog records, predict whether their eating habit is likely to change in the next month, based on patterns in meal_type and meal_name. Return a short prediction.", 3.1667, {"instr_adhere": 4, "accuracy": 4, "clarity": 3, "structure": 3, "match": 3, "reasonable": 2}),
    ("01-ai/Yi-1.5-9B-Chat", "Extra-Large Models", 124, 3.32, "structured_extraction", "From the selected user's UserNote, extract structured information about their eating preferences. Return as key-value pairs.", 2.4, {"instr_adhere": 2, "clarity": 3, "structure": 3, "match": 1, "reasonable": 3}),
    ("01-ai/Yi-1.5-9B-Chat", "Extra-Large Models", 98, 3.32, "structured_extraction", "From the selected user's UserNote, extract structured information about their activity level and exercise habits. Return as key-value pairs.", 4, {"instr_adhere": 4, "clarity": 4, "structure": 4}),
    ("01-ai/Yi-1.5-9B-Chat", "Extra-Large Models", 131, 3.32, "structured_extraction", "From the selected user's UserNote, extract structured information about their fitness goal, including target amount and timeline if mentioned. Return as key-value pairs.", 0, {"instr_adhere": 0, "clarity": 0, "structure": 0}),
    ("01-ai/Yi-1.5-9B-Chat", "Extra-Large Models", 121, 3.32, "structured_extraction", "From the selected user's UserNote, extract any dietary restrictions mentioned. Return as key-value pairs.", 0, {"instr_adhere": 0, "clarity": 0, "structure": 0}),
    ("01-ai/Yi-1.5-9B-Chat", "Extra-Large Models", 128, 3.32, "structured_extraction", "From the selected user's UserNote, extract body condition information that may affect their exercise or diet plan. Return as key-value pairs.", 4, {"instr_adhere": 4, "clarity": 4, "structure": 4}),
    ("01-ai/Yi-1.5-9B-Chat", "Extra-Large Models", 132, 3.47, "summarization", "Based on the selected user's MealLog records, summarize their overall eating behavior in one sentence, covering meal timing, calorie intake, and nutritional balance.", 4, {"instr_adhere": 4, "clarity": 4, "structure": 4, "tone": 4}),
    ("01-ai/Yi-1.5-9B-Chat", "Extra-Large Models", 150, 3.47, "summarization", "Based on the selected user's UserProfile (weight_kg, height_cm, body_fat_pct, activity_level, goal), provide a one-line summary of whether their stated goal is realistic given their current body metrics.", 4, {"instr_adhere": 4, "clarity": 4, "structure": 4, "tone": 4}),
    ("01-ai/Yi-1.5-9B-Chat", "Extra-Large Models", 133, 3.47, "summarization", "Based on the selected user's MealLog records, summarize their diet pattern in one sentence, focusing on meal frequency, meal types, and any noticeable nutritional trends.", 3.25, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 4}),
    ("01-ai/Yi-1.5-9B-Chat", "Extra-Large Models", 129, 3.54, "summarization", "Based on the selected user's MealLog records and BodyFatLog history, summarize their level of progress and consistency in one sentence.", 3, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 3}),
    ("01-ai/Yi-1.5-9B-Chat", "Extra-Large Models", 138, 3.54, "summarization", "Based on the selected user's WeightLog history, BodyFatLog history, and UserProfile (goal, activity_level), summarize their overall progress toward their fitness goal in one sentence.", 3.75, {"instr_adhere": 4, "clarity": 4, "structure": 4, "tone": 3}),
    ("01-ai/Yi-1.5-9B-Chat", "Extra-Large Models", 117, 3.31, "text_generation", "Generate a personalised good morning greeting for a user who is on a fat loss goal.", 3, {"instr_adhere": 3, "clarity": 2, "structure": 3, "tone": 4}),
    ("01-ai/Yi-1.5-9B-Chat", "Extra-Large Models", 125, 3.31, "text_generation", "Generate a short motivational message for a user who has been consistently logging meals but has not seen weight change in 2 weeks.", 3, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 3}),
    ("01-ai/Yi-1.5-9B-Chat", "Extra-Large Models", 115, 3.31, "text_generation", "Generate one practical healthy eating tip suitable for a university student with a busy schedule.", 2.5, {"instr_adhere": 2, "clarity": 3, "structure": 2, "tone": 3}),
    ("01-ai/Yi-1.5-9B-Chat", "Extra-Large Models", 120, 3.31, "text_generation", "Generate a short reminder message for a user who has not logged any meals in the past 3 days.", 3, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 3}),
    ("01-ai/Yi-1.5-9B-Chat", "Extra-Large Models", 115, 3.31, "text_generation", "Generate one follow-up question to better understand why a user keeps skipping breakfast.", 3, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 3}),
    ("upstage/SOLAR-10.7B-Instruct-v1.0", "Extra-Large Models", 103, 10.45, "classification", "Based on the selected user's UserProfile (height_cm, weight_kg, body_fat_pct, age, activity_level), classify their most suitable fitness goal. Possible labels: fat_loss, muscle_gain, maintain. Return only the label.", 2.5, {"instr_adhere": 2, "accuracy": 4, "clarity": 2, "structure": 2}),
    ("upstage/SOLAR-10.7B-Instruct-v1.0", "Extra-Large Models", 160, 19.35, "classification", "Based on the selected user's UserProfile (weight_kg, height_cm, body_fat_pct, activity_level, goal), determine whether their stated goal is appropriate for their current physical condition. Possible labels: match, mismatch. Return only the label.", 3.1667, {"instr_adhere": 3, "accuracy": 3, "clarity": 3, "structure": 3, "match": 4, "reasonable": 3}),
    ("upstage/SOLAR-10.7B-Instruct-v1.0", "Extra-Large Models", 85, 1.41, "classification", "Based on the selected user's MealLog records, classify their eating habit by examining which meal_type entries (breakfast, lunch, dinner) appear or are missing across dates. Possible labels: regular_three_meals, skip_breakfast, skip_lunch, skip_dinner, irregular_meals. Return only the label.", 4.5, {"instr_adhere": 4, "accuracy": 4, "clarity": 5, "structure": 5}),
    ("upstage/SOLAR-10.7B-Instruct-v1.0", "Extra-Large Models", 54, 0.95, "classification", "Based on the selected user's MealLog records, classify their food preference by examining the meal_name entries. Possible labels: asian_only, western_only, salad_heavy, mixed. Return only the label.", 4.5, {"instr_adhere": 5, "accuracy": 3, "clarity": 5, "structure": 5}),
    ("upstage/SOLAR-10.7B-Instruct-v1.0", "Extra-Large Models", 61, 1.41, "classification", "Based on the selected user's MealLog records, classify their app engagement level by counting meals logged per day and the number of distinct dates. Possible labels: low_engagement, moderate_engagement, high_engagement. Return only the label.", 4.5, {"instr_adhere": 5, "accuracy": 3, "clarity": 5, "structure": 5}),
    ("upstage/SOLAR-10.7B-Instruct-v1.0", "Extra-Large Models", 151, 19.04, "prediction", "Based on the selected user's WeightLog history, BodyFatLog history, and UserProfile (goal, activity_level), predict their weight trend over the next 30 days. Return a short prediction with expected direction and estimated change.", 3.6667, {"instr_adhere": 4, "accuracy": 4, "clarity": 4, "structure": 4, "match": 3, "reasonable": 3}),
    ("upstage/SOLAR-10.7B-Instruct-v1.0", "Extra-Large Models", 165, 19.27, "prediction", "Based on the selected user's MealLog (calories, protein_g, carbs_g, fat_g per meal) and UserProfile (goal, activity_level), predict whether their current intake pattern will support their goal over the next 30 days. Return a short prediction.", 3, {"instr_adhere": 3, "accuracy": 3, "clarity": 3, "structure": 3, "match": 3, "reasonable": 3}),
    ("upstage/SOLAR-10.7B-Instruct-v1.0", "Extra-Large Models", 97, 8.89, "prediction", "Based on the selected user's MealLog records, predict whether this user is likely to continue logging meals next week, based on the frequency and recency of their entries. Return: likely or unlikely, with a one-line reason.", 3.5, {"instr_adhere": 4, "accuracy": 3, "clarity": 4, "structure": 4, "match": 3, "reasonable": 3}),
    ("upstage/SOLAR-10.7B-Instruct-v1.0", "Extra-Large Models", 124, 12.73, "prediction", "Based on the selected user's WeightLog trend, BodyFatLog trend, and UserProfile (goal, activity_level), predict whether they are at risk of not achieving their fitness goal. Return: at_risk or on_track, with a one-line reason.", 3.6667, {"instr_adhere": 4, "accuracy": 4, "clarity": 4, "structure": 4, "match": 3, "reasonable": 3}),
    ("upstage/SOLAR-10.7B-Instruct-v1.0", "Extra-Large Models", 144, 24.7, "prediction", "Based on the selected user's MealLog records, predict whether their eating habit is likely to change in the next month, based on patterns in meal_type and meal_name. Return a short prediction.", 3.6667, {"instr_adhere": 4, "accuracy": 4, "clarity": 4, "structure": 4, "match": 3, "reasonable": 3}),
    ("upstage/SOLAR-10.7B-Instruct-v1.0", "Extra-Large Models", 63, 1.98, "structured_extraction", "From the selected user's UserNote, extract structured information about their eating preferences. Return as key-value pairs.", 2, {"instr_adhere": 2, "clarity": 2, "structure": 2}),
    ("upstage/SOLAR-10.7B-Instruct-v1.0", "Extra-Large Models", 54, 1.33, "structured_extraction", "From the selected user's UserNote, extract structured information about their activity level and exercise habits. Return as key-value pairs.", 2.6667, {"instr_adhere": 3, "clarity": 3, "structure": 2}),
    ("upstage/SOLAR-10.7B-Instruct-v1.0", "Extra-Large Models", 75, 2.14, "structured_extraction", "From the selected user's UserNote, extract structured information about their fitness goal, including target amount and timeline if mentioned. Return as key-value pairs.", 4, {"instr_adhere": 4, "clarity": 4, "structure": 4}),
    ("upstage/SOLAR-10.7B-Instruct-v1.0", "Extra-Large Models", 44, 1.08, "structured_extraction", "From the selected user's UserNote, extract any dietary restrictions mentioned. Return as key-value pairs.", 4, {"instr_adhere": 4, "clarity": 4, "structure": 4}),
    ("upstage/SOLAR-10.7B-Instruct-v1.0", "Extra-Large Models", 56, 1.33, "structured_extraction", "From the selected user's UserNote, extract body condition information that may affect their exercise or diet plan. Return as key-value pairs.", 2.3333, {"instr_adhere": 2, "clarity": 3, "structure": 2}),
    ("upstage/SOLAR-10.7B-Instruct-v1.0", "Extra-Large Models", 111, 13.35, "summarization", "Based on the selected user's MealLog records, summarize their overall eating behavior in one sentence, covering meal timing, calorie intake, and nutritional balance.", 4, {"instr_adhere": 4, "clarity": 4, "structure": 4, "tone": 4}),
    ("upstage/SOLAR-10.7B-Instruct-v1.0", "Extra-Large Models", 136, 20.94, "summarization", "Based on the selected user's UserProfile (weight_kg, height_cm, body_fat_pct, activity_level, goal), provide a one-line summary of whether their stated goal is realistic given their current body metrics.", 4, {"instr_adhere": 4, "clarity": 4, "structure": 4, "tone": 4}),
    ("upstage/SOLAR-10.7B-Instruct-v1.0", "Extra-Large Models", 139, 18.6, "summarization", "Based on the selected user's MealLog records, summarize their diet pattern in one sentence, focusing on meal frequency, meal types, and any noticeable nutritional trends.", 3.25, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 4}),
    ("upstage/SOLAR-10.7B-Instruct-v1.0", "Extra-Large Models", 98, 16.46, "summarization", "Based on the selected user's MealLog records and BodyFatLog history, summarize their level of progress and consistency in one sentence.", 3, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 3}),
    ("upstage/SOLAR-10.7B-Instruct-v1.0", "Extra-Large Models", 139, 18.58, "summarization", "Based on the selected user's WeightLog history, BodyFatLog history, and UserProfile (goal, activity_level), summarize their overall progress toward their fitness goal in one sentence.", 3.75, {"instr_adhere": 4, "clarity": 4, "structure": 4, "tone": 3}),
    ("upstage/SOLAR-10.7B-Instruct-v1.0", "Extra-Large Models", 86, 3.14, "text_generation", "Generate a personalised good morning greeting for a user who is on a fat loss goal.", 3.25, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 4}),
    ("upstage/SOLAR-10.7B-Instruct-v1.0", "Extra-Large Models", 116, 4.33, "text_generation", "Generate a short motivational message for a user who has been consistently logging meals but has not seen weight change in 2 weeks.", 3.75, {"instr_adhere": 3, "clarity": 4, "structure": 4, "tone": 4}),
    ("upstage/SOLAR-10.7B-Instruct-v1.0", "Extra-Large Models", 83, 3.07, "text_generation", "Generate one practical healthy eating tip suitable for a university student with a busy schedule.", 3.5, {"instr_adhere": 3, "clarity": 4, "structure": 4, "tone": 3}),
    ("upstage/SOLAR-10.7B-Instruct-v1.0", "Extra-Large Models", 67, 2.11, "text_generation", "Generate a short reminder message for a user who has not logged any meals in the past 3 days.", 3.25, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 4}),
    ("upstage/SOLAR-10.7B-Instruct-v1.0", "Extra-Large Models", 58, 1.8, "text_generation", "Generate one follow-up question to better understand why a user keeps skipping breakfast.", 2.75, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 2}),
    ("lmsys/vicuna-7b-v1.5", "Extra-Large Models", 64, 0.23, "classification", "Based on the selected user's UserProfile (height_cm, weight_kg, body_fat_pct, age, activity_level), classify their most suitable fitness goal. Possible labels: fat_loss, muscle_gain, maintain. Return only the label.", 3.75, {"instr_adhere": 3, "accuracy": 3, "clarity": 4, "structure": 5}),
    ("lmsys/vicuna-7b-v1.5", "Extra-Large Models", 62, 0.28, "classification", "Based on the selected user's UserProfile (weight_kg, height_cm, body_fat_pct, activity_level, goal), determine whether their stated goal is appropriate for their current physical condition. Possible labels: match, mismatch. Return only the label.", 3.25, {"instr_adhere": 4, "accuracy": 1, "clarity": 4, "structure": 4}),
    ("lmsys/vicuna-7b-v1.5", "Extra-Large Models", 85, 0.31, "classification", "Based on the selected user's MealLog records, classify their eating habit by examining which meal_type entries (breakfast, lunch, dinner) appear or are missing across dates. Possible labels: regular_three_meals, skip_breakfast, skip_lunch, skip_dinner, irregular_meals. Return only the label.", 3.5, {"instr_adhere": 4, "accuracy": 4, "clarity": 1, "structure": 5}),
    ("lmsys/vicuna-7b-v1.5", "Extra-Large Models", 57, 0.26, "classification", "Based on the selected user's MealLog records, classify their food preference by examining the meal_name entries. Possible labels: asian_only, western_only, salad_heavy, mixed. Return only the label.", 4, {"instr_adhere": 5, "accuracy": 1, "clarity": 5, "structure": 5}),
    ("lmsys/vicuna-7b-v1.5", "Extra-Large Models", 62, 0.26, "classification", "Based on the selected user's MealLog records, classify their app engagement level by counting meals logged per day and the number of distinct dates. Possible labels: low_engagement, moderate_engagement, high_engagement. Return only the label.", 4.5, {"instr_adhere": 5, "accuracy": 3, "clarity": 5, "structure": 5}),
    ("lmsys/vicuna-7b-v1.5", "Extra-Large Models", 132, 2.32, "prediction", "Based on the selected user's WeightLog history, BodyFatLog history, and UserProfile (goal, activity_level), predict their weight trend over the next 30 days. Return a short prediction with expected direction and estimated change.", 3.3333, {"instr_adhere": 4, "accuracy": 4, "clarity": 4, "structure": 4, "match": 2, "reasonable": 2}),
    ("lmsys/vicuna-7b-v1.5", "Extra-Large Models", 146, 2.37, "prediction", "Based on the selected user's MealLog (calories, protein_g, carbs_g, fat_g per meal) and UserProfile (goal, activity_level), predict whether their current intake pattern will support their goal over the next 30 days. Return a short prediction.", 3, {"instr_adhere": 3, "accuracy": 3, "clarity": 3, "structure": 3, "match": 3, "reasonable": 3}),
    ("lmsys/vicuna-7b-v1.5", "Extra-Large Models", 108, 1.67, "prediction", "Based on the selected user's MealLog records, predict whether this user is likely to continue logging meals next week, based on the frequency and recency of their entries. Return: likely or unlikely, with a one-line reason.", 3.6667, {"instr_adhere": 4, "accuracy": 4, "clarity": 4, "structure": 4, "match": 3, "reasonable": 3}),
    ("lmsys/vicuna-7b-v1.5", "Extra-Large Models", 141, 2.14, "prediction", "Based on the selected user's WeightLog trend, BodyFatLog trend, and UserProfile (goal, activity_level), predict whether they are at risk of not achieving their fitness goal. Return: at_risk or on_track, with a one-line reason.", 3.6667, {"instr_adhere": 4, "accuracy": 4, "clarity": 4, "structure": 4, "match": 3, "reasonable": 3}),
    ("lmsys/vicuna-7b-v1.5", "Extra-Large Models", 146, 2.72, "prediction", "Based on the selected user's MealLog records, predict whether their eating habit is likely to change in the next month, based on patterns in meal_type and meal_name. Return a short prediction.", 3.6667, {"instr_adhere": 4, "accuracy": 4, "clarity": 4, "structure": 4, "match": 3, "reasonable": 3}),
    ("lmsys/vicuna-7b-v1.5", "Extra-Large Models", 102, 1.87, "structured_extraction", "From the selected user's UserNote, extract structured information about their eating preferences. Return as key-value pairs.", 3, {"instr_adhere": 3, "clarity": 3, "structure": 3}),
    ("lmsys/vicuna-7b-v1.5", "Extra-Large Models", 69, 1, "structured_extraction", "From the selected user's UserNote, extract structured information about their activity level and exercise habits. Return as key-value pairs.", 2.6667, {"instr_adhere": 2, "clarity": 2, "structure": 4}),
    ("lmsys/vicuna-7b-v1.5", "Extra-Large Models", 68, 0.82, "structured_extraction", "From the selected user's UserNote, extract structured information about their fitness goal, including target amount and timeline if mentioned. Return as key-value pairs.", 3, {"instr_adhere": 3, "clarity": 2, "structure": 4}),
    ("lmsys/vicuna-7b-v1.5", "Extra-Large Models", 40, 0.4, "structured_extraction", "From the selected user's UserNote, extract any dietary restrictions mentioned. Return as key-value pairs.", 4, {"instr_adhere": 4, "clarity": 4, "structure": 4}),
    ("lmsys/vicuna-7b-v1.5", "Extra-Large Models", 78, 1.2, "structured_extraction", "From the selected user's UserNote, extract body condition information that may affect their exercise or diet plan. Return as key-value pairs.", 4, {"instr_adhere": 4, "clarity": 4, "structure": 4}),
    ("lmsys/vicuna-7b-v1.5", "Extra-Large Models", 141, 2.72, "summarization", "Based on the selected user's MealLog records, summarize their overall eating behavior in one sentence, covering meal timing, calorie intake, and nutritional balance.", 4, {"instr_adhere": 4, "clarity": 4, "structure": 4, "tone": 4}),
    ("lmsys/vicuna-7b-v1.5", "Extra-Large Models", 125, 2.01, "summarization", "Based on the selected user's UserProfile (weight_kg, height_cm, body_fat_pct, activity_level, goal), provide a one-line summary of whether their stated goal is realistic given their current body metrics.", 4, {"instr_adhere": 4, "clarity": 4, "structure": 4, "tone": 4}),
    ("lmsys/vicuna-7b-v1.5", "Extra-Large Models", 144, 2.72, "summarization", "Based on the selected user's MealLog records, summarize their diet pattern in one sentence, focusing on meal frequency, meal types, and any noticeable nutritional trends.", 3.25, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 4}),
    ("lmsys/vicuna-7b-v1.5", "Extra-Large Models", 128, 2.61, "summarization", "Based on the selected user's MealLog records and BodyFatLog history, summarize their level of progress and consistency in one sentence.", 3, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 3}),
    ("lmsys/vicuna-7b-v1.5", "Extra-Large Models", 82, 1.12, "summarization", "Based on the selected user's WeightLog history, BodyFatLog history, and UserProfile (goal, activity_level), summarize their overall progress toward their fitness goal in one sentence.", 3.75, {"instr_adhere": 4, "clarity": 4, "structure": 4, "tone": 3}),
    ("lmsys/vicuna-7b-v1.5", "Extra-Large Models", 32, 0.3, "text_generation", "Generate a personalised good morning greeting for a user who is on a fat loss goal.", 3, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 3}),
    ("lmsys/vicuna-7b-v1.5", "Extra-Large Models", 129, 2.48, "text_generation", "Generate a short motivational message for a user who has been consistently logging meals but has not seen weight change in 2 weeks.", 3, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 3}),
    ("lmsys/vicuna-7b-v1.5", "Extra-Large Models", 119, 2.48, "text_generation", "Generate one practical healthy eating tip suitable for a university student with a busy schedule.", 3.5, {"instr_adhere": 3, "clarity": 4, "structure": 4, "tone": 3}),
    ("lmsys/vicuna-7b-v1.5", "Extra-Large Models", 86, 1.56, "text_generation", "Generate a short reminder message for a user who has not logged any meals in the past 3 days.", 3.25, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 4}),
    ("lmsys/vicuna-7b-v1.5", "Extra-Large Models", 108, 2.23, "text_generation", "Generate one follow-up question to better understand why a user keeps skipping breakfast.", 3, {"instr_adhere": 3, "clarity": 3, "structure": 3, "tone": 3})
]

columns = ["Model", "Model Category", "Tokens/sec", "Latency", "Prompt Type", "Prompt", "Self-Evaluation Score (1–10)"]
df_rows = []
for (model, category, tok_sec, latency, ptype, prompt, score, sub_scores) in BENCHMARK_DATA:
    df_rows.append({
        "Model": model,
        "Model Category": category,
        "Tokens/sec": tok_sec,
        "Latency": latency,
        "Prompt Type": ptype,
        "Prompt": prompt[:80] + "..." if len(prompt) > 80 else prompt,
        "Self-Evaluation Score (1–10)": round(score, 2),
    })

df_benchmarks = pd.DataFrame(df_rows)
print(f"Total benchmark rows: {len(df_benchmarks)}")
print(f"Models: {df_benchmarks['Model'].nunique()}, Prompt Types: {df_benchmarks['Prompt Type'].nunique()}")
print()
df_benchmarks


Total benchmark rows: 375
Models: 15, Prompt Types: 5



,Model,Model Category,Tokens/sec,Latency,Prompt Type,Prompt,Self-Evaluation Score (1–10)
0,Qwen/Qwen2.5-0.5B-Instruct,Ultra-Light Models,133,2.07,classification,"Based on the selected user's UserProfile (height_cm, weight_kg, body_fat_pct...",3.17
1,Qwen/Qwen2.5-0.5B-Instruct,Ultra-Light Models,149,1.88,classification,"Based on the selected user's UserProfile (weight_kg, height_cm, body_fat_pct...",3.67
2,Qwen/Qwen2.5-0.5B-Instruct,Ultra-Light Models,164,1.88,classification,"Based on the selected user's MealLog records, classify their eating habit by...",3.50
3,Qwen/Qwen2.5-0.5B-Instruct,Ultra-Light Models,93,1.00,classification,"Based on the selected user's MealLog records, classify their food preference...",3.00
4,Qwen/Qwen2.5-0.5B-Instruct,Ultra-Light Models,147,1.88,classification,"Based on the selected user's MealLog records, classify their app engagement ...",3.17
5,Qwen/Qwen2.5-0.5B-Instruct,Ultra-Light Models,145,1.88,prediction,"Based on the selected user's WeightLog history, BodyFatLog history, and User...",3.67
6,Qwen/Qwen2.5-0.5B-Instruct,Ultra-Light Models,154,1.88,prediction,"Based on the selected user's MealLog (calories, protein_g, carbs_g, fat_g pe...",2.67
7,Qwen/Qwen2.5-0.5B-Instruct,Ultra-Light Models,147,1.87,prediction,"Based on the selected user's MealLog records, predict whether this user is l...",3.33
8,Qwen/Qwen2.5-0.5B-Instruct,Ultra-Light Models,150,1.88,prediction,"Based on the selected user's WeightLog trend, BodyFatLog trend, and UserProf...",3.67
9,Qwen/Qwen2.5-0.5B-Instruct,Ultra-Light Models,139,1.88,prediction,"Based on the selected user's MealLog records, predict whether their eating h...",3.67


In [5]:
print("=== Average Score by Model Category ===")
cat_order = ["Ultra-Light Models", "Small Models", "Medium Models", "Large Models", "Extra-Large Models"]
cat_stats = df_benchmarks.groupby("Model Category").agg({
    "Self-Evaluation Score (1–10)": "mean",
    "Tokens/sec": "mean",
    "Latency": "mean"
}).reindex(cat_order).round(3)
cat_stats.columns = ["Avg Score", "Avg Tokens/sec", "Avg Latency (s)"]
print(cat_stats.to_string())
print()

print("=== Average Score by Model (sorted by score descending) ===")
model_stats = df_benchmarks.groupby("Model").agg({
    "Self-Evaluation Score (1–10)": "mean",
    "Tokens/sec": "mean",
    "Latency": "mean",
    "Model Category": "first"
}).sort_values("Self-Evaluation Score (1–10)", ascending=False).round(3)
model_stats.columns = ["Avg Score", "Avg Tokens/sec", "Avg Latency (s)", "Category"]
print(model_stats.to_string())
print()

print("=== Average Score by Prompt Type ===")
type_stats = df_benchmarks.groupby("Prompt Type")["Self-Evaluation Score (1–10)"].mean().round(3).sort_values(ascending=False)
print(type_stats.to_string())


=== Average Score by Model Category ===
                    Avg Score  Avg Tokens/sec  Avg Latency (s)
Model Category                                                
Ultra-Light Models      1.913         119.467            1.370
Small Models            2.811         110.280            0.945
Medium Models           3.172         124.933            1.526
Large Models            3.231         106.880            2.047
Extra-Large Models      3.283         111.160            4.692

=== Average Score by Model (sorted by score descending) ===
                                           Avg Score  Avg Tokens/sec  Avg Latency (s)            Category
Model                                                                                                    
lmsys/vicuna-7b-v1.5                           3.470           98.16            1.523  Extra-Large Models
stabilityai/stablelm-zephyr-3b                 3.464           99.68            0.866       Medium Models
upstage/SOLAR-10.7B-Instruct-v1.0 

### Step 1.6: Analysis

In [6]:
print("1. Which model category produced the best outputs?")
print()
best_cat = cat_stats["Avg Score"].idxmax()
print(f"   {best_cat} produced the best outputs with an average score of {cat_stats.loc[best_cat, 'Avg Score']:.3f}.")
print(f"   Extra-Large Models (7B – 10B+ parameters) had the highest average self-evaluation")
print(f"   score across all prompt types.")
print()

print("2. Did larger models always perform better?")
print()
print("   No. Larger models did not always perform better. For example,")
print(f"   01-ai/Yi-1.5-9B-Chat (Extra-Large, 9B) scored {model_stats.loc['01-ai/Yi-1.5-9B-Chat', 'Avg Score']:.3f},")
print(f"   which is lower than every Large model:")
large_models = model_stats[model_stats["Category"] == "Large Models"]
for name, row in large_models.iterrows():
    print(f"     - {name}: {row['Avg Score']:.3f}")
print(f"   Additionally, stabilityai/stablelm-zephyr-3b (Medium, 3B) scored {model_stats.loc['stabilityai/stablelm-zephyr-3b', 'Avg Score']:.3f},")
print(f"   outperforming several larger models.")
print()

print("3. Which model gave the best balance between speed and output quality?")
print()
print("   stabilityai/stablelm-2-zephyr-1_6b gave the best balance between speed and output quality.")
s = model_stats.loc["stabilityai/stablelm-2-zephyr-1_6b"]
print(f"   - Score: {s['Avg Score']:.3f} (ranked 5th out of 15 models)")
print(f"   - Latency: {s['Avg Latency (s)']:.3f}s (fastest of all 15 models)")
print(f"   - Tokens/sec: {s['Avg Tokens/sec']:.2f}")
print(f"   It achieves a competitive quality score while being the fastest model by a wide margin,")
print(f"   making it ideal for latency-sensitive production use.")


1. Which model category produced the best outputs?

   Extra-Large Models produced the best outputs with an average score of 3.283.
   Extra-Large Models (7B – 10B+ parameters) had the highest average self-evaluation
   score across all prompt types.

2. Did larger models always perform better?

   No. Larger models did not always perform better. For example,
   01-ai/Yi-1.5-9B-Chat (Extra-Large, 9B) scored 2.930,
   which is lower than every Large model:
     - NousResearch/Nous-Hermes-2-Mistral-7B-DPO: 3.439
     - HuggingFaceH4/zephyr-7b-beta: 3.217
     - tiiuae/falcon-7b-instruct: 3.036
   Additionally, stabilityai/stablelm-zephyr-3b (Medium, 3B) scored 3.464,
   outperforming several larger models.

3. Which model gave the best balance between speed and output quality?

   stabilityai/stablelm-2-zephyr-1_6b gave the best balance between speed and output quality.
   - Score: 3.257 (ranked 5th out of 15 models)
   - Latency: 0.633s (fastest of all 15 models)
   - Tokens/sec: 93.76
